# 41 — Dataset audit for the 3D/QM descriptor work

**Nothing quantum-mechanical is computed, no conformer is generated, nothing is trained and no
submission is built, validated or sent.** Pure RDKit and pandas over SMILES already on disk.

Everything in the descriptor plan has so far been sized by assumption. This notebook replaces the
assumptions with measurements of the actual compounds, so a conformer protocol, an xtb run and any
ORCA budget can be designed against real numbers rather than guesses.

Run over **both** sets — the curated training set (4,905) and the blind test set (750) — reported
separately and together, and broken down per isoform for the training set. A compound counts toward
every isoform it carries a non-null `pIC50_direct_inhibition` label for, so the per-isoform rows
overlap by construction (26.7% of compounds carry labels for two or more isoforms, per notebook
`05b`).

**Read-only on everything.** `data/`, `src/`, and the outputs and notebooks for `30`, `32` and
`34`–`40` are protected and checked at the end by the three-test scope check notebooks
`30`/`34`/`35`/`37`/`38`/`39` established. `notebooks/32_deadzone_aid_retrain.ipynb` and
`notebooks/40_single_object_ensemble.ipynb` are **never opened or executed** — both contain live
submission code (`32`'s candidate-A cell is uncommented; `40` fired an unintended submission on
2026-10-02). Writes go only to `outputs/41_dataset_audit_3d/`.

## The six sections

1. **Size and cost drivers** — heavy atoms, atoms with explicit H, MW, rotatable bonds, rings,
   aromatic rings. Tails, not means, since QM cost scales superlinearly with size.
2. **Element inventory** — everything present, with anything outside C/N/O/S/P/F/Cl/Br/I named,
   plus formal charge.
3. **Stereochemistry** — read off the **raw** `SMILES` column first, then separately off
   `canonical_smiles`, so that if canonicalisation drops stereo the audit cannot hide it.
4. **Basic nitrogen** — sizes the nitrogen-local descriptor block and tests whether the CYP2D6
   mechanistic story covers enough compounds to be worth computing.
5. **Conformational flexibility** — sizes the conformer protocol, whose cost is conformers ×
   compounds.
6. **Redundancy** — what 3D content, if any, is already in `mordred_pca` and `ecfp4_narrow`.

Then a plain-language verdict on the four questions the brief asks, and the figures that carry them.

In [1]:
import hashlib
import json
import subprocess
import sys
import time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rdkit
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, rdMolDescriptors

from src.vendor.openadmet_eval.config import REGRESSION_ENDPOINTS

ISOFORMS = [e.split("_")[0] for e in REGRESSION_ENDPOINTS]
PIC50_COL = {iso: f"{iso}_pIC50_direct_inhibition" for iso in ISOFORMS}

OUT = REPO_ROOT / "outputs" / "41_dataset_audit_3d"
FIG_OUT = OUT / "figures"
for d in (OUT, FIG_OUT):
    d.mkdir(parents=True, exist_ok=True)

# RUN START recorded at the top, never inferred at the bottom from a file mtime -- the bug
# notebook 38 shipped on its first execution and 30/34/35/37/39 already carry the fix for.
RUN_META_PATH = OUT / "run_meta.json"
if RUN_META_PATH.exists():
    RUN_META = json.loads(RUN_META_PATH.read_text())
    print(f"[resume] run started {RUN_META['started_at_utc']}")
else:
    RUN_META = {"started_at": time.time(),
                "started_at_utc": datetime.now(timezone.utc).isoformat()}
    RUN_META_PATH.write_text(json.dumps(RUN_META, indent=2))
    print(f"[new run] started {RUN_META['started_at_utc']}")
RUN_STARTED_AT = float(RUN_META["started_at"])

# READ-ONLY sources.
CURATED = REPO_ROOT / "data" / "processed" / "train_inhibition_curated.csv"
BLIND = REPO_ROOT / "data" / "processed" / "test_blinded_curated.csv"
RAW_TRAIN = REPO_ROOT / "data" / "raw" / "cyp-challenge-TRAIN_inhibition.csv"
RAW_BLIND = REPO_ROOT / "data" / "raw" / "cyp-challenge-TEST-BLINDED.csv"
ECFP4_NARROW = REPO_ROOT / "data" / "processed" / "tabular_baseline_features.csv"
MORDRED_PCA = REPO_ROOT / "data" / "processed" / "tabular_mordred_pca.csv"
MORDRED_NAN_REPORT = REPO_ROOT / "outputs" / "mordred_pca" / "descriptor_nan_report.csv"
NB17_CHARGE = REPO_ROOT / "outputs" / "17_protonation_charge_check" / "charge_logp_features.csv"
NB23_POOL = REPO_ROOT / "outputs" / "23_compound_pool_overlap" / "pool_overlap_counts.csv"

DECISIONS = []


def record_decision(decision, reason, alternatives, authority):
    DECISIONS.append({"decision": decision, "reason": reason, "alternatives": alternatives,
                      "authority": authority, "at_utc": datetime.now(timezone.utc).isoformat()})
    (OUT / "decisions_taken.json").write_text(json.dumps(DECISIONS, indent=2))


def pct(n, d):
    return 100.0 * n / d if d else float("nan")


# Every RDKit warning is surfaced, not swallowed: nothing here should emit one, and if the audit
# starts warning on real compounds that is itself a finding.
RDLogger.EnableLog("rdApp.*")

print(f"rdkit {rdkit.__version__} | pandas {pd.__version__} | numpy {np.__version__}")
print(f"python {sys.version.split()[0]}")
print(f"isoforms: {ISOFORMS}")
print(f"writing to {OUT.relative_to(REPO_ROOT)}/")

[new run] started 2026-10-03T11:49:28.665708+00:00
rdkit 2026.03.3 | pandas 2.3.3 | numpy 1.26.4
python 3.11.13
isoforms: ['CYP1A2', 'CYP2C9', 'CYP2D6', 'CYP3A4']
writing to outputs/41_dataset_audit_3d/


### Part 0 — provenance: the environment, the sources, and the per-isoform counts

Four things checked before a single descriptor is computed.

**The RDKit version**, because this notebook's stereochemistry section depends on which API means
what. `CLAUDE.md` pins `rdkit==2026.3.3` in `cyp-admet-v2` and records a full-row determinism check
against it; this notebook asserts it is actually running under that version rather than assuming the
kernel was started in the right environment.

**The two input files**, by sha256 and row count, so the numbers below are traceable to exact bytes.

**The per-isoform scored counts**, computed here and cross-checked against notebook `23`'s own saved
`pool_overlap_counts.csv` — the project's established reference for them. Verifying against the
saved file rather than against prose is this project's convention, and it has caught three quoting
errors in its own history.

**One RDKit semantic**, established by direct test rather than from documentation: in this version,
`CalcNumAtomStereoCenters` counts **specified and unspecified centres together**. That matters twice
over — Section 3 needs the two apart, and `ecfp4_narrow`'s own `stereocenter_count` column is built
from exactly this call, so Section 6 has to say what that column actually holds.

In [2]:
# ---- (1) environment.
PINNED_RDKIT = "2026.03.3"
assert rdkit.__version__ == PINNED_RDKIT, (
    f"expected rdkit {PINNED_RDKIT} (cyp-admet-v2's pin, per CLAUDE.md) but got "
    f"{rdkit.__version__} -- the stereo API semantics below were established against the pin")
print(f"rdkit {rdkit.__version__} matches CLAUDE.md's pin for cyp-admet-v2.")

# ---- (2) sources, by bytes.
prov = []
for label, path in [("curated train", CURATED), ("blind test", BLIND),
                    ("raw train (source)", RAW_TRAIN), ("raw blind (source)", RAW_BLIND)]:
    h = hashlib.sha256(path.read_bytes()).hexdigest()
    n = len(pd.read_csv(path))
    prov.append({"label": label, "path": str(path.relative_to(REPO_ROOT)), "rows": n, "sha256": h})
    print(f"  {label:20s} {n:5d} rows  sha256 {h[:16]}...  {path.relative_to(REPO_ROOT)}")
prov = pd.DataFrame(prov)

train = pd.read_csv(CURATED)
blind = pd.read_csv(BLIND)
assert len(train) == 4905 and len(blind) == 750, (len(train), len(blind))
assert train["SMILES"].notna().all() and blind["SMILES"].notna().all()
assert train["Molecule_Name"].is_unique and blind["Molecule_Name"].is_unique

# ---- (3) per-isoform scored counts, cross-checked against notebook 23's saved file.
counts_here = {iso: int(train[PIC50_COL[iso]].notna().sum()) for iso in ISOFORMS}
nb23 = pd.read_csv(NB23_POOL)
counts_nb23 = (nb23.drop_duplicates("isoform").set_index("isoform")["existing_scored_pool"]
               .astype(int).to_dict())
print("\nper-isoform scored label counts:")
ok = True
for iso in ISOFORMS:
    same = counts_here[iso] == counts_nb23[iso]
    ok &= same
    print(f"  {iso}  here {counts_here[iso]:5d}   notebook 23 {counts_nb23[iso]:5d}   "
          f"{'MATCH' if same else '*** MISMATCH'}")
assert ok, "per-isoform counts disagree with notebook 23's saved reference -- stop and investigate"
n_multi = int((train[[PIC50_COL[i] for i in ISOFORMS]].notna().sum(axis=1) >= 2).sum())
print(f"  total measurements {sum(counts_here.values())} over {len(train)} compounds; "
      f"{n_multi} compounds ({pct(n_multi, len(train)):.1f}%) carry 2+ isoform labels")

# ---- (4) the RDKit stereo-counter semantic, established by test not by documentation.
_unspec = Chem.MolFromSmiles("CC(N)C(=O)O")          # one UNSPECIFIED centre
_spec = Chem.MolFromSmiles("C[C@H](N)C(=O)O")        # the same centre, SPECIFIED
_flat = Chem.MolFromSmiles("CCO")                    # no centre at all
sem = pd.DataFrame([
    {"smiles": s,
     "CalcNumAtomStereoCenters": rdMolDescriptors.CalcNumAtomStereoCenters(m),
     "CalcNumUnspecified": rdMolDescriptors.CalcNumUnspecifiedAtomStereoCenters(m),
     "FindPotentialStereo": "; ".join(
         f"{str(e.type).split('.')[-1]}={str(e.specified).split('.')[-1]}"
         for e in Chem.FindPotentialStereo(m)) or "(none)"}
    for s, m in [("CC(N)C(=O)O", _unspec), ("C[C@H](N)C(=O)O", _spec), ("CCO", _flat)]])
print("\nRDKit stereo-counter semantics, by direct test:")
print(sem.to_string(index=False))
assert rdMolDescriptors.CalcNumAtomStereoCenters(_unspec) == 1, (
    "CalcNumAtomStereoCenters did NOT count an unspecified centre -- Section 6's reading of "
    "ecfp4_narrow's stereocenter_count column depends on this and must be redone")
print("\nCONFIRMED: CalcNumAtomStereoCenters counts specified AND unspecified centres together.")
print("  -> Section 3 uses FindPotentialStereo, which reports the two separately.")
print("  -> Section 6: ecfp4_narrow's `stereocenter_count` is a POTENTIAL-centre count, not an")
print("     assigned-configuration count. Recorded there.")
prov.to_csv(OUT / "part0_provenance.csv", index=False)

rdkit 2026.03.3 matches CLAUDE.md's pin for cyp-admet-v2.
  curated train         4905 rows  sha256 9209ecd45ea8945b...  data/processed/train_inhibition_curated.csv
  blind test             750 rows  sha256 ba5e6ee298485c5d...  data/processed/test_blinded_curated.csv
  raw train (source)    4905 rows  sha256 b8f79addd266fb6f...  data/raw/cyp-challenge-TRAIN_inhibition.csv
  raw blind (source)     750 rows  sha256 a342f8444a8dcb53...  data/raw/cyp-challenge-TEST-BLINDED.csv

per-isoform scored label counts:
  CYP1A2  here  1412   notebook 23  1412   MATCH
  CYP2C9  here  1285   notebook 23  1285   MATCH
  CYP2D6  here  1493   notebook 23  1493   MATCH
  CYP3A4  here  2335   notebook 23  2335   MATCH
  total measurements 6525 over 4905 compounds; 1309 compounds (26.7%) carry 2+ isoform labels

RDKit stereo-counter semantics, by direct test:
         smiles  CalcNumAtomStereoCenters  CalcNumUnspecified          FindPotentialStereo
    CC(N)C(=O)O                         1                 

### Part 0.1 — one pass over every compound

Every per-compound quantity the six sections need is computed once here, into a single table, and
every section then reads off it. Both sets go through the identical code path, so a train/blind
difference below can never be an artefact of two different computations.

Two deliberate choices, both recorded in "Decisions taken without approval":

- **Stereo is read off the raw `SMILES` column**, as Section 3's brief requires, with
  `canonical_smiles` computed separately in Section 3 for the comparison. Everything else is read
  off raw `SMILES` too, for consistency — the two forms describe the same molecule, and Section 3
  confirms that directly rather than assuming it.
- **Rotatable bonds use RDKit's `strict=True` default.** The non-strict count is carried alongside,
  because the two differ by about one bond per compound and a conformer budget sized against the
  wrong one is off by a multiplicative factor.

In [3]:
BASIC_N_STRICT = (
    "[NX3;H0,H1,H2;!+"            # trivalent, neutral nitrogen
    ";!$(N[C,S,P]=[O,N,S])"       # not amide / thioamide / sulfonamide / phosphoramide
    ";!$(N[S,P]=O)"               # not sulfonyl / phosphoryl nitrogen
    ";!$(N#*);!$(N=*)"            # not nitrile / imine / azo nitrogen
    ";!$(N[a])"                   # not aniline-type (conjugated into a ring, weakly basic)
    ";!$(NC=[!#6])"
    ";!$([nX3])]"                 # not aromatic
)
BASIC_N_LOOSE = BASIC_N_STRICT.replace(";!$(N[a])", "")   # same, but aniline-type N allowed
P_STRICT = Chem.MolFromSmarts(BASIC_N_STRICT)
P_LOOSE = Chem.MolFromSmarts(BASIC_N_LOOSE)
assert P_STRICT is not None and P_LOOSE is not None, "basic-N SMARTS failed to compile"

ALLOWED_ELEMENTS = {"C", "N", "O", "S", "P", "F", "Cl", "Br", "I"}
UNSPEC = Chem.StereoSpecified.Unspecified
SPEC = Chem.StereoSpecified.Specified


def audit_one(smiles):
    # Every per-compound quantity the audit needs, from one parse of `smiles`.
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    mol_h = Chem.AddHs(mol)
    stereo = Chem.FindPotentialStereo(mol)
    tetra = [e for e in stereo if e.type == Chem.StereoType.Atom_Tetrahedral]
    dbl = [e for e in stereo if e.type == Chem.StereoType.Bond_Double]
    symbols = [a.GetSymbol() for a in mol.GetAtoms()]
    return {
        # --- Section 1: size and cost drivers
        "heavy_atoms": mol.GetNumHeavyAtoms(),
        "total_atoms_with_h": mol_h.GetNumAtoms(),
        "mol_wt": Descriptors.MolWt(mol),
        "rotatable_bonds": rdMolDescriptors.CalcNumRotatableBonds(mol, strict=True),
        "rotatable_bonds_nonstrict": rdMolDescriptors.CalcNumRotatableBonds(mol, strict=False),
        "ring_count": rdMolDescriptors.CalcNumRings(mol),
        "aromatic_ring_count": rdMolDescriptors.CalcNumAromaticRings(mol),
        # --- Section 2: elements, charge, open-shell character
        "elements": "|".join(sorted(set(symbols))),
        "n_elements": len(set(symbols)),
        "has_nonstandard_element": bool(set(symbols) - ALLOWED_ELEMENTS),
        "nonstandard_elements": "|".join(sorted(set(symbols) - ALLOWED_ELEMENTS)),
        "formal_charge": Chem.GetFormalCharge(mol),
        "n_radical_electrons": sum(a.GetNumRadicalElectrons() for a in mol.GetAtoms()),
        "n_charged_atoms": sum(1 for a in mol.GetAtoms() if a.GetFormalCharge() != 0),
        "n_fragments": len(Chem.GetMolFrags(mol)),
        # --- Section 3: stereochemistry
        "n_stereocentres": len(tetra),
        "n_stereocentres_unassigned": sum(1 for e in tetra if e.specified == UNSPEC),
        "n_stereocentres_assigned": sum(1 for e in tetra if e.specified == SPEC),
        "n_double_bond_stereo_potential": len(dbl),
        "n_double_bond_stereo_defined": sum(1 for e in dbl if e.specified == SPEC),
        # --- Section 4: basic nitrogen
        "n_basic_n": len(mol.GetSubstructMatches(P_STRICT)),
        "n_basic_n_loose": len(mol.GetSubstructMatches(P_LOOSE)),
        "n_nitrogen": sum(1 for s in symbols if s == "N"),
    }


rows = []
for set_name, df in [("train", train), ("blind", blind)]:
    for r in df.itertuples(index=False):
        rec = audit_one(r.SMILES)
        assert rec is not None, f"SMILES failed to parse: {r.Molecule_Name} {r.SMILES}"
        rec["set"] = set_name
        rec["Molecule_Name"] = r.Molecule_Name
        rec["inchikey"] = r.inchikey
        rec["SMILES"] = r.SMILES
        for iso in ISOFORMS:
            rec[f"has_{iso}"] = (set_name == "train"
                                 and bool(pd.notna(getattr(r, PIC50_COL[iso]))))
        rows.append(rec)

AUD = pd.DataFrame(rows)
front = ["set", "Molecule_Name", "inchikey"]
AUD = AUD[front + [c for c in AUD.columns if c not in front + ["SMILES"]] + ["SMILES"]]
assert len(AUD) == 4905 + 750, len(AUD)
assert AUD.notna().all().all(), "unexpected NaN in the audit table"
for iso in ISOFORMS:
    assert int(AUD[AUD.set == "train"][f"has_{iso}"].sum()) == counts_here[iso]
AUD.to_csv(OUT / "per_compound_audit.csv", index=False)
print(f"audited {len(AUD)} compounds ({(AUD.set == 'train').sum()} train + "
      f"{(AUD.set == 'blind').sum()} blind), 0 parse failures, "
      f"{AUD.shape[1]} columns -> per_compound_audit.csv")
print(f"multi-fragment (salt/mixture) compounds: {int((AUD.n_fragments > 1).sum())}")

TR = AUD[AUD.set == "train"].reset_index(drop=True)
BL = AUD[AUD.set == "blind"].reset_index(drop=True)
POPS = ([("train (all)", TR), ("blind (all)", BL), ("both sets", AUD)]
        + [(f"train / {iso}", TR[TR[f"has_{iso}"]]) for iso in ISOFORMS])
for name, pop in POPS:
    print(f"  population {name:18s} n = {len(pop)}")

audited 5655 compounds (4905 train + 750 blind), 0 parse failures, 31 columns -> per_compound_audit.csv
multi-fragment (salt/mixture) compounds: 0
  population train (all)        n = 4905
  population blind (all)        n = 750
  population both sets          n = 5655
  population train / CYP1A2     n = 1412
  population train / CYP2C9     n = 1285
  population train / CYP2D6     n = 1493
  population train / CYP3A4     n = 2335


### Part 0.2 — what each number is for

Stated once, so the tables below read as a budget rather than as chemistry trivia.

| quantity | what it sizes |
|---|---|
| heavy atoms | GFN2-xTB cost (roughly `N²`–`N³`); the DFT single-point cost a later ORCA step would carry (closer to `N³`–`N⁴`) |
| atoms with explicit H | the system size an optimiser actually sees — hydrogens are free in a 2D descriptor and are not free in a geometry optimisation |
| rotatable bonds | conformers per compound, and so the multiplier on everything else |
| rings / aromatic rings | rigidity, and where ring-pucker conformers are needed |
| elements | GFN2 parameterisation coverage, and whether a basis set can be chosen automatically |
| formal charge / radicals | whether a job is closed-shell neutral (the easy case) or needs charge and multiplicity set by hand |
| stereocentres, assigned vs not | whether a 3D structure is determined by the data at all |

---

## Section 1 — size and cost drivers

Six quantities per compound, summarised as mean, median, the 50th/90th/95th/99th percentiles and
the max, for every population: both sets together, each set alone, and each isoform within the
training set.

**The tail is what matters here, not the mean.** A GFN2 optimisation scales roughly as `N²`–`N³` in
heavy atoms and a DFT single point closer to `N³`–`N⁴`, so doubling the size of a compound can cost
an order of magnitude. A budget built on the mean will be wrong in the direction that hurts.

In [4]:
SIZE_COLS = ["heavy_atoms", "total_atoms_with_h", "mol_wt", "rotatable_bonds",
             "ring_count", "aromatic_ring_count"]
STATS = ["mean", "median", "p50", "p90", "p95", "p99", "max"]


def summarise(pop, cols):
    out = []
    for c in cols:
        v = pop[c].to_numpy(dtype=float)
        out.append({"quantity": c, "n": len(v), "mean": v.mean(), "median": float(np.median(v)),
                    "p50": float(np.percentile(v, 50)), "p90": float(np.percentile(v, 90)),
                    "p95": float(np.percentile(v, 95)), "p99": float(np.percentile(v, 99)),
                    "max": v.max(), "min": v.min(), "sd": v.std(ddof=1)})
    return pd.DataFrame(out)


size_tbl = []
for name, pop in POPS:
    s = summarise(pop, SIZE_COLS)
    s.insert(0, "population", name)
    size_tbl.append(s)
size_tbl = pd.concat(size_tbl, ignore_index=True)
size_tbl.to_csv(OUT / "section1_size_summary.csv", index=False)

for q in SIZE_COLS:
    sub = size_tbl[size_tbl.quantity == q].set_index("population")
    print(f"\n{q}")
    print(sub[["n"] + STATS].round(2).to_string())
print(f"\n-> section1_size_summary.csv ({len(size_tbl)} rows: "
      f"{len(POPS)} populations x {len(SIZE_COLS)} quantities)")


heavy_atoms
                   n   mean  median   p50   p90   p95    p99   max
population                                                        
train (all)     4905  24.62    24.0  24.0  29.0  30.0  34.00  65.0
blind (all)      750  23.87    24.0  24.0  28.0  30.0  34.00  39.0
both sets       5655  24.52    24.0  24.0  29.0  30.0  34.00  65.0
train / CYP1A2  1412  24.04    24.0  24.0  28.0  29.0  32.89  53.0
train / CYP2C9  1285  25.22    25.0  25.0  29.0  32.0  35.00  46.0
train / CYP2D6  1493  24.77    24.0  24.0  29.0  31.0  34.08  65.0
train / CYP3A4  2335  24.65    24.0  24.0  29.0  30.0  33.00  46.0

total_atoms_with_h
                   n   mean  median   p50   p90   p95    p99    max
population                                                         
train (all)     4905  44.93    45.0  45.0  54.0  56.0  62.96  126.0
blind (all)      750  42.46    42.0  42.0  54.0  58.0  62.00   88.0
both sets       5655  44.60    44.0  44.0  54.0  57.0  62.46  126.0
train / CYP1A2  1412  42

#### 1.1 — the large-compound threshold counts, and the ten biggest

How many compounds sit above 50, 60 and 70 heavy atoms, per population. These are the compounds
that would dominate any QM budget, and the ones where SCF convergence problems cluster — so they
are named individually, with their SMILES, rather than left as a count.

In [5]:
thr_rows = []
for name, pop in POPS:
    r = {"population": name, "n": len(pop)}
    for t in (50, 60, 70):
        k = int((pop.heavy_atoms > t).sum())
        r[f"n_gt_{t}_HA"] = k
        r[f"pct_gt_{t}_HA"] = pct(k, len(pop))
    r["max_HA"] = int(pop.heavy_atoms.max())
    r["max_atoms_with_H"] = int(pop.total_atoms_with_h.max())
    thr_rows.append(r)
thr = pd.DataFrame(thr_rows)
thr.to_csv(OUT / "section1_large_compound_counts.csv", index=False)
print("compounds above each heavy-atom threshold:")
print(thr.round(3).to_string(index=False))

TOPN = 10
largest = (AUD.nlargest(TOPN, "heavy_atoms")
           [["set", "Molecule_Name", "heavy_atoms", "total_atoms_with_h", "mol_wt",
             "rotatable_bonds", "ring_count", "aromatic_ring_count", "formal_charge",
             "n_stereocentres", "n_stereocentres_unassigned", "elements", "SMILES"]]
           .reset_index(drop=True))
largest.insert(0, "rank", range(1, len(largest) + 1))
isoform_of = []
for nm in largest.Molecule_Name:
    row = AUD[AUD.Molecule_Name == nm].iloc[0]
    isoform_of.append("|".join(i for i in ISOFORMS if row[f"has_{i}"]) or "(blind)")
largest["isoform_labels"] = isoform_of
largest.to_csv(OUT / "section1_largest_compounds.csv", index=False)
print(f"\nthe {TOPN} largest compounds across both sets:")
print(largest[["rank", "set", "Molecule_Name", "heavy_atoms", "total_atoms_with_h", "mol_wt",
               "rotatable_bonds", "isoform_labels"]].round(1).to_string(index=False))
print("\nSMILES:")
for r in largest.itertuples():
    print(f"  {r.rank:2d}. {r.Molecule_Name}  ({r.heavy_atoms} HA, {r.total_atoms_with_h} atoms "
          f"with H)\n      {r.SMILES}")

# The same list for each set alone, since the blind set is what gets scored.
for set_name in ("train", "blind"):
    sub = AUD[AUD.set == set_name].nlargest(5, "heavy_atoms")
    print(f"\ntop 5 within {set_name}: "
          + ", ".join(f"{r.Molecule_Name} ({r.heavy_atoms} HA)" for r in sub.itertuples()))

compounds above each heavy-atom threshold:
    population    n  n_gt_50_HA  pct_gt_50_HA  n_gt_60_HA  pct_gt_60_HA  n_gt_70_HA  pct_gt_70_HA  max_HA  max_atoms_with_H
   train (all) 4905           3         0.061           1         0.020           0           0.0      65               126
   blind (all)  750           0         0.000           0         0.000           0           0.0      39                88
     both sets 5655           3         0.053           1         0.018           0           0.0      65               126
train / CYP1A2 1412           1         0.071           0         0.000           0           0.0      53               119
train / CYP2C9 1285           0         0.000           0         0.000           0           0.0      46                87
train / CYP2D6 1493           2         0.134           1         0.067           0           0.0      65               126
train / CYP3A4 2335           0         0.000           0         0.000           0      

#### 1.2 — which isoform carries the largest compounds, and what the top 1% looks like

The brief asks this explicitly. Answered on the p99 and the max together, because with per-isoform
populations of 1,285–2,335 the max is a single compound and the p99 is the honest tail statistic.

In [6]:
iso_tail = []
for iso in ISOFORMS:
    pop = TR[TR[f"has_{iso}"]]
    top1 = pop[pop.heavy_atoms >= np.percentile(pop.heavy_atoms, 99)]
    iso_tail.append({
        "isoform": iso, "n": len(pop),
        "HA_mean": pop.heavy_atoms.mean(), "HA_p95": float(np.percentile(pop.heavy_atoms, 95)),
        "HA_p99": float(np.percentile(pop.heavy_atoms, 99)), "HA_max": int(pop.heavy_atoms.max()),
        "n_in_top1pct": len(top1),
        "top1pct_HA_mean": top1.heavy_atoms.mean(),
        "top1pct_atoms_with_H_mean": top1.total_atoms_with_h.mean(),
        "top1pct_MW_mean": top1.mol_wt.mean(),
        "top1pct_rotb_mean": top1.rotatable_bonds.mean(),
        "n_gt_50_HA": int((pop.heavy_atoms > 50).sum()),
        "largest_compound": pop.nlargest(1, "heavy_atoms").Molecule_Name.iloc[0]})
iso_tail = pd.DataFrame(iso_tail)
iso_tail.to_csv(OUT / "section1_isoform_tails.csv", index=False)
print("per-isoform size tail (training set):")
print(iso_tail.round(2).to_string(index=False))

worst_p99 = iso_tail.loc[iso_tail.HA_p99.idxmax()]
worst_max = iso_tail.loc[iso_tail.HA_max.idxmax()]
spread_p99 = iso_tail.HA_p99.max() - iso_tail.HA_p99.min()
print(f"\nLARGEST COMPOUNDS BY ISOFORM")
print(f"  on p99:  {worst_p99.isoform} ({worst_p99.HA_p99:.1f} HA) -- but the spread across all "
      f"four isoforms is only {spread_p99:.1f} HA")
print(f"  on max:  {worst_max.isoform} ({int(worst_max.HA_max)} HA, {worst_max.largest_compound})")

# What the whole set's own top 1% looks like, both sets together.
cut = float(np.percentile(AUD.heavy_atoms, 99))
top1_all = AUD[AUD.heavy_atoms >= cut]
print(f"\nTHE TOP 1% OF BOTH SETS COMBINED (heavy atoms >= {cut:.0f}): n = {len(top1_all)}")
print(f"  heavy atoms        {top1_all.heavy_atoms.min()}-{top1_all.heavy_atoms.max()} "
      f"(mean {top1_all.heavy_atoms.mean():.1f}) vs {AUD.heavy_atoms.mean():.1f} overall")
print(f"  atoms with H       {top1_all.total_atoms_with_h.min()}-"
      f"{top1_all.total_atoms_with_h.max()} (mean {top1_all.total_atoms_with_h.mean():.1f})")
print(f"  MW                 {top1_all.mol_wt.min():.0f}-{top1_all.mol_wt.max():.0f}")
print(f"  rotatable bonds    {top1_all.rotatable_bonds.min()}-{top1_all.rotatable_bonds.max()} "
      f"(mean {top1_all.rotatable_bonds.mean():.1f})")
print(f"  of which blind     {int((top1_all.set == 'blind').sum())} of {len(top1_all)}")

per-isoform size tail (training set):
isoform    n  HA_mean  HA_p95  HA_p99  HA_max  n_in_top1pct  top1pct_HA_mean  top1pct_atoms_with_H_mean  top1pct_MW_mean  top1pct_rotb_mean  n_gt_50_HA largest_compound
 CYP1A2 1412    24.04    29.0   32.89      53            15            36.60                      66.33           537.09               8.67           1     OCNT-2328510
 CYP2C9 1285    25.22    32.0   35.00      46            14            37.71                      66.36           515.64               6.14           0     OCNT-2328402
 CYP2D6 1493    24.77    31.0   34.08      65            15            41.40                      77.60           578.87               9.67           2     OCNT-2328942
 CYP3A4 2335    24.65    30.0   33.00      46            38            34.68                      62.05           477.36               7.21           0     OCNT-2328402

LARGEST COMPOUNDS BY ISOFORM
  on p99:  CYP2C9 (35.0 HA) -- but the spread across all four isoforms is only 2.1 HA
 

#### 1.3 — what this actually implies for a QM budget

A cost model, stated explicitly so it can be argued with rather than assumed. GFN2 wall time is
taken as scaling with heavy atoms to the power 2.5 — between the quadratic and cubic bounds, and
the exponent only enters as a ratio here, so the conclusion is checked against 2.0 and 3.0 as well.

The question is whether the tail dominates. It does if a small number of large compounds account
for a large share of total predicted cost.

In [7]:
def cost_share(pop, exponent):
    c = pop.heavy_atoms.to_numpy(float) ** exponent
    c = np.sort(c)[::-1]
    tot = c.sum()
    out = {}
    for k_pct in (1, 5, 10):
        k = max(1, int(round(len(c) * k_pct / 100)))
        out[f"top{k_pct}pct_share"] = c[:k].sum() / tot
    out["cost_vs_median_compound"] = (c.max() /
                                      (np.median(pop.heavy_atoms.to_numpy(float)) ** exponent))
    return out


rows = []
for exponent in (2.0, 2.5, 3.0, 4.0):
    for name, pop in [("both sets", AUD), ("train (all)", TR), ("blind (all)", BL)]:
        r = {"scaling_exponent": exponent, "population": name, "n": len(pop)}
        r.update(cost_share(pop, exponent))
        rows.append(r)
cost_tbl = pd.DataFrame(rows)
cost_tbl.to_csv(OUT / "section1_cost_model.csv", index=False)
print("predicted share of total cost carried by the largest compounds:")
print(cost_tbl.round(4).to_string(index=False))

ref = cost_tbl[(cost_tbl.scaling_exponent == 2.5) & (cost_tbl.population == "both sets")].iloc[0]
print(f"""
READ AT exponent 2.5, both sets (5,655 compounds):
  the largest  1% carries {ref['top1pct_share'] * 100:.1f}% of predicted total cost
  the largest  5% carries {ref['top5pct_share'] * 100:.1f}%
  the largest 10% carries {ref['top10pct_share'] * 100:.1f}%
  the single largest compound costs {ref['cost_vs_median_compound']:.1f}x the median compound.

For scale: if cost were flat across compounds, the top 1% would carry exactly 1.0%, the top 5%
5.0% and the top 10% 10.0%. The observed shares are the honest measure of how skewed the budget is.
""")

predicted share of total cost carried by the largest compounds:
 scaling_exponent  population    n  top1pct_share  top5pct_share  top10pct_share  cost_vs_median_compound
              2.0   both sets 5655         0.0239         0.0905          0.1606                   7.3351
              2.0 train (all) 4905         0.0240         0.0899          0.1594                   7.3351
              2.0 blind (all)  750         0.0227         0.0948          0.1666                   2.6406
              2.5   both sets 5655         0.0298         0.1046          0.1799                  12.0713
              2.5 train (all) 4905         0.0302         0.1039          0.1785                  12.0713
              2.5 blind (all)  750         0.0271         0.1093          0.1871                   3.3661
              3.0   both sets 5655         0.0374         0.1209          0.2013                  19.8658
              3.0 train (all) 4905         0.0381         0.1202          0.1997        

---

## Section 2 — element inventory

Every element present in either set, counted both as atoms and as the number of compounds
containing it. Anything outside C/N/O/S/P/F/Cl/Br/I is named explicitly with the compounds
involved, because GFN2's parameterisation is thinner there and a basis set stops being chooseable
by default. Boron, silicon, selenium, tin and any metal would all land in that bucket.

Formal charge and radical electrons are reported in the same section: a non-neutral or open-shell
species needs charge and multiplicity set by hand rather than defaulted, which is the difference
between one job script and two.

In [8]:
el_rows = []
for name, pop in POPS:
    per_compound = Counter()
    per_atom = Counter()
    for r in pop.itertuples():
        mol = Chem.MolFromSmiles(r.SMILES)
        syms = [a.GetSymbol() for a in mol.GetAtoms()]
        per_atom.update(syms)
        per_compound.update(set(syms))
    for el in sorted(per_compound, key=lambda e: -per_compound[e]):
        el_rows.append({"population": name, "n_population": len(pop), "element": el,
                        "n_compounds": per_compound[el],
                        "pct_compounds": pct(per_compound[el], len(pop)),
                        "n_atoms": per_atom[el],
                        "outside_CNOSP_halogen": el not in ALLOWED_ELEMENTS})
el_tbl = pd.DataFrame(el_rows)
el_tbl.to_csv(OUT / "section2_element_inventory.csv", index=False)

for name in ["both sets", "train (all)", "blind (all)"]:
    sub = el_tbl[el_tbl.population == name]
    print(f"\n{name} (n={sub.n_population.iloc[0]}):")
    print(sub[["element", "n_compounds", "pct_compounds", "n_atoms",
               "outside_CNOSP_halogen"]].round(2).to_string(index=False))

print("\nper-isoform element presence (% of that isoform's compounds):")
wide = (el_tbl[el_tbl.population.str.startswith("train / ")]
        .pivot(index="element", columns="population", values="pct_compounds"))
print(wide.reindex(el_tbl[el_tbl.population == "train (all)"].element).round(1).to_string())


both sets (n=5655):
element  n_compounds  pct_compounds  n_atoms  outside_CNOSP_halogen
      C         5655         100.00   101203                  False
      N         5586          98.78    19812                  False
      O         5382          95.17    12398                  False
      S         1553          27.46     1745                  False
      F         1372          24.26     2532                  False
     Cl          669          11.83      761                  False
     Br          161           2.85      163                  False
      I           11           0.19       24                  False
      P            3           0.05        3                  False

train (all) (n=4905):
element  n_compounds  pct_compounds  n_atoms  outside_CNOSP_halogen
      C         4905         100.00    88183                  False
      N         4837          98.61    17348                  False
      O         4646          94.72    10710                  False
    

In [9]:
# ---- the flag that matters: anything outside C/N/O/S/P/F/Cl/Br/I.
offenders = AUD[AUD.has_nonstandard_element]
print("=" * 84)
print("ELEMENTS OUTSIDE C/N/O/S/P/F/Cl/Br/I")
print("=" * 84)
all_present = set(el_tbl[el_tbl.population == "both sets"].element)
outside = sorted(all_present - ALLOWED_ELEMENTS)
print(f"  elements present across both sets: {sorted(all_present)}")
print(f"  outside the allowed set:           {outside if outside else 'NONE'}")
print(f"  compounds involved:                {len(offenders)}")
if len(offenders):
    print(offenders[["set", "Molecule_Name", "nonstandard_elements", "SMILES"]].to_string(index=False))
else:
    print("\n  Nothing to flag. No boron, silicon, selenium, tin, arsenic or any metal appears in")
    print("  either set, so GFN2's parameterisation covers every atom present and a basis set can")
    print("  be chosen from the element list without a per-compound exception.")
offenders.to_csv(OUT / "section2_nonstandard_element_compounds.csv", index=False)

# Radicals / open-shell.
rad = AUD[AUD.n_radical_electrons > 0]
print(f"\nRADICAL / OPEN-SHELL SPECIES: {len(rad)}")
if len(rad):
    print(rad[["set", "Molecule_Name", "n_radical_electrons", "SMILES"]].to_string(index=False))
else:
    print("  none -- every compound in both sets is closed-shell as written, so multiplicity 1")
    print("  is correct throughout and no unrestricted calculation is needed.")

# The heaviest element present is the one that actually constrains basis-set choice.
iodine = AUD[[("I" in e.split("|")) for e in AUD.elements]]
print(f"\nHEAVIEST ELEMENT PRESENT: iodine, in {len(iodine)} compounds "
      f"({int((iodine.set == 'train').sum())} train + {int((iodine.set == 'blind').sum())} blind).")
print("  Inside GFN2's parameterisation and inside the allowed set, so not a flag -- but it is the")
print("  one element here where an all-electron DFT step would need a relativistic ECP rather than")
print("  a default basis, so the compounds are named.")
print(iodine[["set", "Molecule_Name", "heavy_atoms", "SMILES"]].to_string(index=False))
iodine.to_csv(OUT / "section2_iodine_compounds.csv", index=False)

phos = AUD[[("P" in e.split("|")) for e in AUD.elements]]
print(f"\nphosphorus: {len(phos)} compounds "
      f"({int((phos.set == 'train').sum())} train + {int((phos.set == 'blind').sum())} blind)")
print(phos[["set", "Molecule_Name", "SMILES"]].to_string(index=False))

ELEMENTS OUTSIDE C/N/O/S/P/F/Cl/Br/I
  elements present across both sets: ['Br', 'C', 'Cl', 'F', 'I', 'N', 'O', 'P', 'S']
  outside the allowed set:           NONE
  compounds involved:                0

  Nothing to flag. No boron, silicon, selenium, tin, arsenic or any metal appears in
  either set, so GFN2's parameterisation covers every atom present and a basis set can
  be chosen from the element list without a per-compound exception.

RADICAL / OPEN-SHELL SPECIES: 0
  none -- every compound in both sets is closed-shell as written, so multiplicity 1
  is correct throughout and no unrestricted calculation is needed.

HEAVIEST ELEMENT PRESENT: iodine, in 11 compounds (7 train + 4 blind).
  Inside GFN2's parameterisation and inside the allowed set, so not a flag -- but it is the
  one element here where an all-electron DFT step would need a relativistic ECP rather than
  a default basis, so the compounds are named.
  set Molecule_Name  heavy_atoms                                     

In [10]:
# ---- formal charge.
print("=" * 84)
print("FORMAL CHARGE")
print("=" * 84)
chg_rows = []
for name, pop in POPS:
    chg_rows.append({"population": name, "n": len(pop),
                     "n_neutral": int((pop.formal_charge == 0).sum()),
                     "n_nonneutral": int((pop.formal_charge != 0).sum()),
                     "pct_nonneutral": pct(int((pop.formal_charge != 0).sum()), len(pop)),
                     "n_cationic": int((pop.formal_charge > 0).sum()),
                     "n_anionic": int((pop.formal_charge < 0).sum()),
                     "min_charge": int(pop.formal_charge.min()),
                     "max_charge": int(pop.formal_charge.max())})
chg = pd.DataFrame(chg_rows)
chg.to_csv(OUT / "section2_formal_charge_summary.csv", index=False)
print(chg.round(2).to_string(index=False))

nn = AUD[AUD.formal_charge != 0].copy()
# A permanently charged centre (quaternary N, iminium) is a different problem from a carboxylate
# written deprotonated: the first MUST be run as an ion, the second is a drawing convention.
QUAT = Chem.MolFromSmarts("[NX4+,nX3+,#7+;!$([NX4+][OX1-])]")
CARBOXYLATE = Chem.MolFromSmarts("[CX3](=O)[OX1-]")
kinds = []
for r in nn.itertuples():
    mol = Chem.MolFromSmiles(r.SMILES)
    has_quat = mol.HasSubstructMatch(QUAT)
    has_carbox = mol.HasSubstructMatch(CARBOXYLATE)
    kinds.append("permanent cation (quaternary N / iminium)" if has_quat
                 else "carboxylate written deprotonated" if has_carbox
                 else "other")
nn["charge_kind"] = kinds
nn.to_csv(OUT / "section2_nonneutral_compounds.csv", index=False)
print(f"\nthe {len(nn)} non-neutral compounds, named:")
print(nn[["set", "Molecule_Name", "formal_charge", "n_charged_atoms", "charge_kind",
          "SMILES"]].to_string(index=False))
print("\nby kind:")
print(nn.groupby(["charge_kind", "formal_charge"]).size().to_string())
print(f"""
WHY THE SPLIT MATTERS FOR A QM SETUP
  A permanent cation (a quaternary ammonium, an iminium, a cyanine) genuinely carries its charge
  and must be run with charge set accordingly -- there is no neutral form of it.
  A carboxylate written as [O-] is a drawing convention in this file; the same compound could as
  correctly have been written neutral, and notebook 17 already established that 55.4% of training
  compounds change charge state under an explicit pH 7.4 protonation model. So the count of
  non-neutral compounds here is a count of how the SMILES were WRITTEN, not of how many compounds
  are charged at physiological pH -- notebook 17's `net_charge_ph74` is the quantity for that, and
  Section 4 cross-checks against it.
""")

FORMAL CHARGE
    population    n  n_neutral  n_nonneutral  pct_nonneutral  n_cationic  n_anionic  min_charge  max_charge
   train (all) 4905       4884            21            0.43          11         10          -1           1
   blind (all)  750        750             0            0.00           0          0           0           0
     both sets 5655       5634            21            0.37          11         10          -1           1
train / CYP1A2 1412       1403             9            0.64           6          3          -1           1
train / CYP2C9 1285       1281             4            0.31           0          4          -1           0
train / CYP2D6 1493       1483            10            0.67           9          1          -1           1
train / CYP3A4 2335       2327             8            0.34           3          5          -1           1

the 21 non-neutral compounds, named:
  set Molecule_Name  formal_charge  n_charged_atoms                               ch

---

## Section 3 — stereochemistry

**This section decides whether a chirality angle exists at all**, so the order of operations is
deliberate and is the one the brief specifies.

Stereo is read off the **raw `SMILES` column of the curated CSV**, directly, with nothing routed
through `src/features.py`'s canonicalisation first. If that pipeline stripped stereochemistry,
auditing through it would hide exactly what is being measured. Only then is the same count run on
`canonical_smiles` and the two compared, as an independent check on whether canonicalisation
preserves or drops stereo.

Two further guards:

- The counts are also run on the **raw source files** (`data/raw/cyp-challenge-TRAIN_inhibition.csv`
  and `cyp-challenge-TEST-BLINDED.csv`) rather than only the curated derivatives, so that if
  curation lost stereo this notebook says so instead of concluding the compounds are flat.
- Three counts are reported separately, not merged: **potential** stereocentres (where a centre
  exists), **unassigned** centres (where the data does not say which configuration), and
  **assigned** centres. Part 0 established that RDKit's `CalcNumAtomStereoCenters` merges the first
  and third, which is why `FindPotentialStereo` is used here.

In [11]:
def stereo_of(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    st = Chem.FindPotentialStereo(mol)
    tetra = [e for e in st if e.type == Chem.StereoType.Atom_Tetrahedral]
    dbl = [e for e in st if e.type == Chem.StereoType.Bond_Double]
    return {"n_centres": len(tetra),
            "n_unassigned": sum(1 for e in tetra if e.specified == UNSPEC),
            "n_assigned": sum(1 for e in tetra if e.specified == SPEC),
            "n_db_potential": len(dbl),
            "n_db_defined": sum(1 for e in dbl if e.specified == SPEC)}


raw_train = pd.read_csv(RAW_TRAIN)
raw_blind = pd.read_csv(RAW_BLIND)

# Four SMILES sources per set: the raw source file, the curated raw SMILES column, the curated
# canonical_smiles column, and canonicalisation recomputed live through src/features.py.
from src.features import canonicalize_smiles

SOURCES = {
    ("train", "raw source file"): raw_train.set_index("Molecule_Name").loc[
        train.Molecule_Name, "SMILES"].to_numpy(),
    ("train", "curated SMILES (raw)"): train.SMILES.to_numpy(),
    ("train", "curated canonical_smiles"): train.canonical_smiles.to_numpy(),
    ("train", "canonicalised live via src/features.py"): np.array(
        [canonicalize_smiles(s) for s in train.SMILES]),
    ("blind", "raw source file"): raw_blind.set_index("Molecule_Name").loc[
        blind.Molecule_Name, "SMILES"].to_numpy(),
    ("blind", "curated SMILES (raw)"): blind.SMILES.to_numpy(),
    ("blind", "curated canonical_smiles"): blind.canonical_smiles.to_numpy(),
    ("blind", "canonicalised live via src/features.py"): np.array(
        [canonicalize_smiles(s) for s in blind.SMILES]),
}

st_rows = []
ST_BY_SOURCE = {}
for (set_name, src_name), smis in SOURCES.items():
    assert pd.notna(smis).all(), f"null SMILES in {set_name}/{src_name}"
    d = pd.DataFrame([stereo_of(s) for s in smis])
    assert d.notna().all().all(), f"parse failure in {set_name}/{src_name}"
    ST_BY_SOURCE[(set_name, src_name)] = d
    n = len(d)
    st_rows.append({
        "set": set_name, "smiles_source": src_name, "n": n,
        "n_with_stereo_markers": int(pd.Series(smis).str.contains("@", regex=False).sum()),
        "n_ge1_centre": int((d.n_centres >= 1).sum()),
        "pct_ge1_centre": pct(int((d.n_centres >= 1).sum()), n),
        "n_ge1_unassigned": int((d.n_unassigned >= 1).sum()),
        "pct_ge1_unassigned": pct(int((d.n_unassigned >= 1).sum()), n),
        "n_fully_specified": int(((d.n_centres >= 1) & (d.n_unassigned == 0)).sum()),
        "pct_fully_specified": pct(int(((d.n_centres >= 1) & (d.n_unassigned == 0)).sum()), n),
        "n_gt1_centre": int((d.n_centres > 1).sum()),
        "total_centres": int(d.n_centres.sum()),
        "total_assigned_centres": int(d.n_assigned.sum()),
        "max_centres": int(d.n_centres.max()),
        "n_db_stereo_defined": int((d.n_db_defined >= 1).sum()),
        "total_db_stereo_defined": int(d.n_db_defined.sum())})
st_tbl = pd.DataFrame(st_rows)
st_tbl.to_csv(OUT / "section3_stereo_by_source.csv", index=False)
print("stereochemistry, counted independently off four SMILES sources per set:")
print(st_tbl[["set", "smiles_source", "n", "n_with_stereo_markers", "n_ge1_centre",
              "n_ge1_unassigned", "n_fully_specified", "n_gt1_centre", "total_centres",
              "total_assigned_centres", "n_db_stereo_defined"]].to_string(index=False))

stereochemistry, counted independently off four SMILES sources per set:
  set                          smiles_source    n  n_with_stereo_markers  n_ge1_centre  n_ge1_unassigned  n_fully_specified  n_gt1_centre  total_centres  total_assigned_centres  n_db_stereo_defined
train                        raw source file 4905                    535          2458              1999                459           773           3552                    1068                   19
train                   curated SMILES (raw) 4905                    535          2458              1999                459           773           3552                    1068                   19
train               curated canonical_smiles 4905                    535          2458              1999                459           773           3552                    1068                   19
train canonicalised live via src/features.py 4905                    535          2458              1999                459           77

#### 3.1 — does canonicalisation preserve stereo?

The check the brief asks for, answered against the table above and then at the level of individual
compounds — a per-compound comparison, not just matching aggregates, since two different compounds
gaining and losing a centre would cancel in a total.

In [12]:
print("=" * 84)
print("DOES CANONICALISATION PRESERVE STEREOCHEMISTRY?")
print("=" * 84)
canon_rows = []
for set_name in ("train", "blind"):
    raw = ST_BY_SOURCE[(set_name, "curated SMILES (raw)")]
    for other in ("raw source file", "curated canonical_smiles",
                  "canonicalised live via src/features.py"):
        o = ST_BY_SOURCE[(set_name, other)]
        diffs = {c: int((raw[c] != o[c]).sum()) for c in raw.columns}
        canon_rows.append({"set": set_name, "compared_against": other,
                           "n": len(raw), "compounds_differing_in_any_count":
                               int((raw != o).any(axis=1).sum()), **diffs})
canon = pd.DataFrame(canon_rows)
canon.to_csv(OUT / "section3_canonicalisation_check.csv", index=False)
print(canon.to_string(index=False))

n_diff = int(canon.compounds_differing_in_any_count.sum())
if n_diff == 0:
    print(f"""
VERDICT: canonicalisation PRESERVES stereochemistry, exactly.
  Across all {len(train) + len(blind)} compounds and both directions of comparison, not a single
  compound's potential-centre, unassigned-centre, assigned-centre or double-bond-stereo count
  changes between the raw SMILES, the raw source file, the stored `canonical_smiles` column, and
  `src/features.py`'s `canonicalize_smiles` recomputed live. RDKit's `MolToSmiles` emits isomeric
  SMILES by default and this project's canonicalisation does not override that.
  So the warning behind this section's instruction does not bite here -- but the audit was still
  run off the raw column, and the equivalence is a measurement rather than an assumption.""")
else:
    print(f"\n*** VERDICT: canonicalisation CHANGES stereo on {n_diff} compound comparisons -- "
          f"every stereo figure in this notebook must be read off the raw column only.")

print(f"""
AND THE SOURCE FILE AGREES.
  The raw challenge files carry identical counts to the curated derivatives, so curation did not
  strip stereochemistry either. The low assigned-centre counts below are a property of the released
  data, not of anything this project did to it.""")

DOES CANONICALISATION PRESERVE STEREOCHEMISTRY?
  set                       compared_against    n  compounds_differing_in_any_count  n_centres  n_unassigned  n_assigned  n_db_potential  n_db_defined
train                        raw source file 4905                                 0          0             0           0               0             0
train               curated canonical_smiles 4905                                 0          0             0           0               0             0
train canonicalised live via src/features.py 4905                                 0          0             0           0               0             0
blind                        raw source file  750                                 0          0             0           0               0             0
blind               curated canonical_smiles  750                                 0          0             0           0               0             0
blind canonicalised live via src/features.py  

#### 3.2 — the three counts the brief asks for, and the per-isoform breakdown

How many compounds have at least one stereocentre, at least one **unassigned** stereocentre, and
more than one stereocentre. Read off the raw `SMILES` column, per the instruction.

In [13]:
stereo_rows = []
for name, pop in POPS:
    n = len(pop)
    ge1 = int((pop.n_stereocentres >= 1).sum())
    un = int((pop.n_stereocentres_unassigned >= 1).sum())
    full = int(((pop.n_stereocentres >= 1) & (pop.n_stereocentres_unassigned == 0)).sum())
    gt1 = int((pop.n_stereocentres > 1).sum())
    stereo_rows.append({
        "population": name, "n": n,
        "n_no_centre": n - ge1, "pct_no_centre": pct(n - ge1, n),
        "n_ge1_centre": ge1, "pct_ge1_centre": pct(ge1, n),
        "n_ge1_unassigned": un, "pct_ge1_unassigned": pct(un, n),
        "n_fully_specified": full, "pct_fully_specified": pct(full, n),
        "n_gt1_centre": gt1, "pct_gt1_centre": pct(gt1, n),
        "total_centres": int(pop.n_stereocentres.sum()),
        "total_unassigned_centres": int(pop.n_stereocentres_unassigned.sum()),
        "total_assigned_centres": int(pop.n_stereocentres_assigned.sum()),
        "max_centres": int(pop.n_stereocentres.max()),
        "n_db_stereo_defined": int((pop.n_double_bond_stereo_defined >= 1).sum()),
        # the share of compounds WITH a centre whose configuration is nowhere specified
        "pct_of_centred_compounds_unassigned": pct(un, ge1) if ge1 else float("nan")})
stereo_tbl = pd.DataFrame(stereo_rows)
stereo_tbl.to_csv(OUT / "section3_stereo_summary.csv", index=False)
print(stereo_tbl[["population", "n", "n_no_centre", "n_ge1_centre", "n_ge1_unassigned",
                  "n_fully_specified", "n_gt1_centre", "max_centres",
                  "n_db_stereo_defined"]].to_string(index=False))
print("\nas percentages of each population:")
print(stereo_tbl[["population", "n", "pct_no_centre", "pct_ge1_centre", "pct_ge1_unassigned",
                  "pct_fully_specified", "pct_gt1_centre",
                  "pct_of_centred_compounds_unassigned"]].round(1).to_string(index=False))

tr_row = stereo_tbl[stereo_tbl.population == "train (all)"].iloc[0]
bl_row = stereo_tbl[stereo_tbl.population == "blind (all)"].iloc[0]
print("=" * 84)
print("THE NUMBER THAT DECIDES THE CHIRALITY QUESTION")
print("=" * 84)
for label, row in [("TRAINING SET", tr_row), ("BLIND SET (the compounds that get scored)", bl_row)]:
    print(f"\n  {label} -- {int(row['n'])} compounds")
    print(f"    at least one potential stereocentre   {int(row['n_ge1_centre']):5d}  "
          f"({row['pct_ge1_centre']:5.1f}%)")
    print(f"    at least one UNASSIGNED centre        {int(row['n_ge1_unassigned']):5d}  "
          f"({row['pct_ge1_unassigned']:5.1f}%)")
    print(f"    every centre specified                {int(row['n_fully_specified']):5d}  "
          f"({row['pct_fully_specified']:5.1f}%)")
    print(f"    more than one centre                  {int(row['n_gt1_centre']):5d}  "
          f"({row['pct_gt1_centre']:5.1f}%)")
_bc, _bu = int(bl_row["n_ge1_centre"]), int(bl_row["n_ge1_unassigned"])
print()
print(f"  Of the {_bc} blind compounds that HAVE a stereocentre, {_bu} "
      f"({bl_row['pct_of_centred_compounds_unassigned']:.1f}%) do not say which configuration")
print(f"  was tested. Only {int(bl_row['n_fully_specified'])} of the {int(bl_row['n'])} blind "
      f"compounds -- {bl_row['pct_fully_specified']:.1f}% -- have every centre they possess")
print("  specified.")

    population    n  n_no_centre  n_ge1_centre  n_ge1_unassigned  n_fully_specified  n_gt1_centre  max_centres  n_db_stereo_defined
   train (all) 4905         2447          2458              1999                459           773           14                   19
   blind (all)  750          484           266               248                 18            81            6                    0
     both sets 5655         2931          2724              2247                477           854           14                   19
train / CYP1A2 1412          823           589               486                103           142            8                    6
train / CYP2C9 1285          644           641               523                118           200           11                    6
train / CYP2D6 1493          686           807               655                152           280           14                    6
train / CYP3A4 2335         1186          1149               928            

---

## Section 4 — basic nitrogen

Counts basic nitrogens per compound: aliphatic nitrogen, not amide, not aromatic, with a lone pair
available. This sizes the nitrogen-local descriptor block and tests whether the CYP2D6 mechanistic
story — a protonated basic amine pairing with Asp301 — covers enough compounds to be worth
computing. The per-isoform breakdown is the point, since that argument is isoform-specific.

**The definition is a judgement call the brief does not pin down**, so two things are done rather
than one. A specific SMARTS is written out, commented clause by clause, and **validated against ten
reference compounds before being applied** — four of them notebook `17`'s own CYP2D6-relevant basic
amines, reused verbatim, plus two of its acids and four more chosen to probe the amide and aromatic
exclusions specifically. And a **looser** variant, differing only in whether aniline-type nitrogen
counts, is carried through every table as a sensitivity band, so the conclusion can be seen to
survive the definitional choice or not.

Both are then cross-checked against notebook `17`'s saved pH 7.4 protonation states — an
independent instrument (Dimorphite-DL's pKa model, not a SMARTS) on the same compounds.

In [14]:
# Ten reference compounds. The first six are notebook 17's own Part 1 spot-check list, reused
# verbatim (including its dihydroquinidine SMILES for "quinidine"); the last four are added here to
# probe the amide and aromatic exclusions, which notebook 17 had no reason to test.
BASIC_N_REFERENCE = [
    ("propranolol", "CC(C)NCC(O)COc1cccc2ccccc12", 1, "secondary aliphatic amine"),
    ("quinidine", "CC[C@H]1CN2CC[C@H]1C[C@@H]2[C@H](O)c1ccnc2ccc(OC)cc12", 1,
     "bridged tertiary amine; the quinoline N is aromatic and must NOT count"),
    ("desipramine", "CNCCCN1c2ccccc2CCc2ccccc21", 1,
     "one secondary amine; the ring N is aniline-type and must NOT count under the strict rule"),
    ("metoprolol", "COCCc1ccc(OCC(O)CNC(C)C)cc1", 1, "secondary aliphatic amine"),
    ("diclofenac", "OC(=O)Cc1ccccc1Nc1c(Cl)cccc1Cl", 0, "acid; its N is a diarylamine"),
    ("warfarin", "CC(=O)CC(c1ccccc1)c1c(O)c2ccccc2oc1=O", 0, "acid, no nitrogen at all"),
    ("nicotine", "CN1CCC[C@H]1c1cccnc1", 1,
     "pyrrolidine N counts; the pyridine N is aromatic and must NOT"),
    ("lidocaine", "CCN(CC)CC(=O)Nc1c(C)cccc1C", 1,
     "tertiary amine counts; the anilide N is an amide and must NOT"),
    ("acetanilide", "CC(=O)Nc1ccccc1", 0, "amide only -- tests the amide exclusion"),
    ("caffeine", "Cn1c(=O)c2c(ncn2C)n(C)c1=O", 0,
     "all nitrogens aromatic or amide -- tests both exclusions at once"),
]

ref_rows = []
for nm, smi, expected_ge1, why in BASIC_N_REFERENCE:
    mol = Chem.MolFromSmiles(smi)
    assert mol is not None, nm
    strict = len(mol.GetSubstructMatches(P_STRICT))
    loose = len(mol.GetSubstructMatches(P_LOOSE))
    ok = (strict >= 1) == (expected_ge1 >= 1)
    ref_rows.append({"compound": nm, "expect_ge1_basic_N": expected_ge1,
                     "n_basic_N_strict": strict, "n_basic_N_loose": loose,
                     "PASS": ok, "what it tests": why})
ref = pd.DataFrame(ref_rows)
ref.to_csv(OUT / "section4_smarts_validation.csv", index=False)
print("basic-N SMARTS validation, before the definition is applied to any real compound:")
print(ref[["compound", "expect_ge1_basic_N", "n_basic_N_strict", "n_basic_N_loose",
           "PASS"]].to_string(index=False))
n_pass = int(ref.PASS.sum())
print(f"\n  {n_pass}/{len(ref)} PASS")
assert n_pass == len(ref), "basic-N SMARTS failed its own reference check -- do not apply it"
print("  strict SMARTS:", BASIC_N_STRICT)
print("  loose SMARTS :", BASIC_N_LOOSE)
print("\n  Note desipramine and nicotine: both carry a second nitrogen the strict rule excludes")
print("  (aniline-type and aromatic respectively), and the loose variant picks desipramine's up --")
print("  which is exactly the definitional axis the sensitivity band below measures.")

basic-N SMARTS validation, before the definition is applied to any real compound:
   compound  expect_ge1_basic_N  n_basic_N_strict  n_basic_N_loose  PASS
propranolol                   1                 1                1  True
  quinidine                   1                 1                1  True
desipramine                   1                 1                2  True
 metoprolol                   1                 1                1  True
 diclofenac                   0                 0                1  True
   warfarin                   0                 0                0  True
   nicotine                   1                 1                1  True
  lidocaine                   1                 1                1  True
acetanilide                   0                 0                0  True
   caffeine                   0                 0                0  True

  10/10 PASS
  strict SMARTS: [NX3;H0,H1,H2;!+;!$(N[C,S,P]=[O,N,S]);!$(N[S,P]=O);!$(N#*);!$(N=*);!$(N[a]);!$(NC=[!

In [15]:
bn_rows = []
for name, pop in POPS:
    n = len(pop)
    r = {"population": name, "n": n}
    for tag, col in [("strict", "n_basic_n"), ("loose", "n_basic_n_loose")]:
        v = pop[col].to_numpy()
        r.update({f"n_zero_{tag}": int((v == 0).sum()),
                  f"n_one_{tag}": int((v == 1).sum()),
                  f"n_two_plus_{tag}": int((v >= 2).sum()),
                  f"n_ge1_{tag}": int((v >= 1).sum()),
                  f"pct_ge1_{tag}": pct(int((v >= 1).sum()), n),
                  f"mean_{tag}": float(v.mean()), f"max_{tag}": int(v.max())})
    r["n_with_any_nitrogen"] = int((pop.n_nitrogen >= 1).sum())
    bn_rows.append(r)
bn = pd.DataFrame(bn_rows)
bn.to_csv(OUT / "section4_basic_nitrogen_summary.csv", index=False)
print("basic nitrogen -- none, one, or more (strict definition), with the loose band beside it:")
print(bn[["population", "n", "n_zero_strict", "n_one_strict", "n_two_plus_strict",
          "pct_ge1_strict", "pct_ge1_loose", "mean_strict", "max_strict",
          "n_with_any_nitrogen"]].round(2).to_string(index=False))

iso_bn = bn[bn.population.str.startswith("train / ")].copy()
iso_bn["isoform"] = [p.split(" / ")[1] for p in iso_bn.population]
others = iso_bn[iso_bn.isoform != "CYP2D6"]
d6_row = iso_bn[iso_bn.isoform == "CYP2D6"].iloc[0]
print("=" * 84)
print("THE PER-ISOFORM RESULT -- the CYP2D6 argument tested against this project's own data")
print("=" * 84)
print(iso_bn[["isoform", "n", "n_ge1_strict", "pct_ge1_strict", "mean_strict",
              "n_ge1_loose", "pct_ge1_loose", "mean_loose"]].round(2).to_string(index=False))
print()
print(f"  CYP2D6:      {d6_row['pct_ge1_strict']:.1f}% of its {int(d6_row['n'])} labelled "
      f"compounds carry at least one basic nitrogen "
      f"(loose: {d6_row['pct_ge1_loose']:.1f}%)")
print(f"  other three: {others.pct_ge1_strict.min():.1f}-{others.pct_ge1_strict.max():.1f}% "
      f"(loose: {others.pct_ge1_loose.min():.1f}-{others.pct_ge1_loose.max():.1f}%)")
print(f"  enrichment:  {d6_row['pct_ge1_strict'] / others.pct_ge1_strict.mean():.2f}x the mean of "
      f"the other three under the strict rule, "
      f"{d6_row['pct_ge1_loose'] / others.pct_ge1_loose.mean():.2f}x under the loose rule.")
print()
print("  The enrichment survives the definitional choice, which is the point of carrying both.")
print("  This is an independent, structural confirmation of the CYP2D6 story at the level of which")
print("  compounds were put on the assay -- distinct from, and consistent with, notebook 17's")
print("  finding that CYP2D6 is the one isoform whose charge-pIC50 correlation is NOT explained by")
print("  a lipophilicity confound.")

basic nitrogen -- none, one, or more (strict definition), with the loose band beside it:
    population    n  n_zero_strict  n_one_strict  n_two_plus_strict  pct_ge1_strict  pct_ge1_loose  mean_strict  max_strict  n_with_any_nitrogen
   train (all) 4905           3979           864                 62           18.88          36.17         0.20           2                 4837
   blind (all)  750            663            82                  5           11.60          19.47         0.12           2                  749
     both sets 5655           4642           946                 67           17.91          33.95         0.19           2                 5586
train / CYP1A2 1412           1228           172                 12           13.03          34.14         0.14           2                 1388
train / CYP2C9 1285           1122           152                 11           12.68          30.04         0.14           2                 1260
train / CYP2D6 1493            935       

In [16]:
# ---- cross-check against notebook 17's saved pH 7.4 protonation states.
nb17 = pd.read_csv(NB17_CHARGE)
assert len(nb17) == 5655, len(nb17)
xc = AUD[["set", "inchikey", "Molecule_Name", "n_basic_n", "n_basic_n_loose", "n_nitrogen"]].merge(
    nb17[["inchikey", "net_charge_ph74", "crippen_logp"]], on="inchikey", how="left")
print(f"matched {int(xc.net_charge_ph74.notna().sum())} of {len(xc)} compounds to notebook 17's "
      f"saved charge file (by inchikey)")
xc = xc[xc.net_charge_ph74.notna()]
cationic = xc.net_charge_ph74 >= 1

rows = []
for tag, col in [("strict", "n_basic_n"), ("loose", "n_basic_n_loose")]:
    has = xc[col] >= 1
    rows.append({"definition": tag,
                 "n": len(xc),
                 "n_basic_N_and_cationic": int((has & cationic).sum()),
                 "n_basic_N_not_cationic": int((has & ~cationic).sum()),
                 "n_no_basic_N_but_cationic": int((~has & cationic).sum()),
                 "n_no_basic_N_not_cationic": int((~has & ~cationic).sum()),
                 "agreement_pct": 100.0 * float((has == cationic).mean())})
agree = pd.DataFrame(rows)
agree.to_csv(OUT / "section4_nb17_crosscheck.csv", index=False)
print("\nSMARTS basic-N (>=1) against notebook 17's Dimorphite-DL net charge (>= +1 at pH 7.4):")
print(agree.round(1).to_string(index=False))

print(f"""
WHAT THE DISAGREEMENT IS, STATED PLAINLY RATHER THAN AVERAGED AWAY.
  The strict rule agrees with notebook 17's protonation model on
  {agree.loc[0, 'agreement_pct']:.1f}% of compounds, the loose rule on
  {agree.loc[1, 'agreement_pct']:.1f}%. The residual is not noise and is not symmetric: under the
  strict rule {int(agree.loc[0, 'n_no_basic_N_but_cationic'])} compounds carry NO basic nitrogen by
  this SMARTS yet protonate to +1 or more under Dimorphite-DL.

  These are two different measurements, not two attempts at the same one. This section counts
  structural basic centres of a specific kind -- the kind a nitrogen-local descriptor block would
  be computed ON. Notebook 17 asked a different question: what charge does a pKa model assign to
  the whole molecule at pH 7.4, over its own broader set of ionisable patterns. A compound can be
  cationic there through a centre this SMARTS deliberately excludes.

  So the strict count is best read as a LOWER BOUND on compounds with a protonatable basic centre,
  and notebook 17's cationic fraction as the upper one. Both are reported; neither is adjusted
  toward the other.
""")

# Characterise the disagreeing compounds structurally, with RDKit alone, rather than hand-waving.
dis = xc[(xc.n_basic_n == 0) & cationic]
probes = {"aromatic N present": "[n]",
          "aniline-type N present": "[NX3;!$(N[C,S,P]=[O,N,S])][a]",
          "amidine or guanidine present": "[NX3][CX3]=[NX2]",
          "any aliphatic NX3 present": "[NX3;!$([n])]"}
print(f"structure of the {len(dis)} strict-zero-but-cationic compounds:")
for label, sm in probes.items():
    p = Chem.MolFromSmarts(sm)
    k = sum(1 for s in dis.Molecule_Name
            if Chem.MolFromSmiles(AUD.set_index('Molecule_Name').loc[s, 'SMILES']).HasSubstructMatch(p))
    print(f"  {label:32s} {k:5d} of {len(dis)}  ({pct(k, len(dis)):.1f}%)")
dis.to_csv(OUT / "section4_crosscheck_disagreements.csv", index=False)

matched 5655 of 5655 compounds to notebook 17's saved charge file (by inchikey)

SMARTS basic-N (>=1) against notebook 17's Dimorphite-DL net charge (>= +1 at pH 7.4):
definition    n  n_basic_N_and_cationic  n_basic_N_not_cationic  n_no_basic_N_but_cationic  n_no_basic_N_not_cationic  agreement_pct
    strict 5655                     888                     125                       1433                       3209           72.4
     loose 5655                    1215                     705                       1106                       2629           68.0

WHAT THE DISAGREEMENT IS, STATED PLAINLY RATHER THAN AVERAGED AWAY.
  The strict rule agrees with notebook 17's protonation model on
  72.4% of compounds, the loose rule on
  68.0%. The residual is not noise and is not symmetric: under the
  strict rule 1433 compounds carry NO basic nitrogen by
  this SMARTS yet protonate to +1 or more under Dimorphite-DL.

  These are two different measurements, not two attempts at the same one

  aromatic N present                 999 of 1433  (69.7%)


  aniline-type N present             329 of 1433  (23.0%)


  amidine or guanidine present        22 of 1433  (1.5%)


  any aliphatic NX3 present         1426 of 1433  (99.5%)


---

## Section 5 — conformational flexibility

The rotatable-bond distribution, and the count of compounds in the 0–2, 3–5, 6–10 and >10 bins.
This sizes the conformer protocol directly: a rigid compound needs few conformers and a floppy one
needs many, and the cost of the whole pipeline is conformers × compounds.

Both the strict and non-strict RDKit counts are reported. They differ by about one bond per
compound on average, which propagates into a conformer budget as a multiplicative factor, so the
choice is made explicitly rather than inherited from a default.

In [17]:
BINS = [(0, 2, "0-2 (rigid)"), (3, 5, "3-5"), (6, 10, "6-10"), (11, 10 ** 6, ">10 (floppy)")]

flex_rows = []
for name, pop in POPS:
    n = len(pop)
    r = {"population": name, "n": n}
    for lo, hi, label in BINS:
        k = int(((pop.rotatable_bonds >= lo) & (pop.rotatable_bonds <= hi)).sum())
        r[f"n_{label}"] = k
        r[f"pct_{label}"] = pct(k, n)
    r.update({"rotb_mean": pop.rotatable_bonds.mean(),
              "rotb_median": float(np.median(pop.rotatable_bonds)),
              "rotb_p90": float(np.percentile(pop.rotatable_bonds, 90)),
              "rotb_p95": float(np.percentile(pop.rotatable_bonds, 95)),
              "rotb_p99": float(np.percentile(pop.rotatable_bonds, 99)),
              "rotb_max": int(pop.rotatable_bonds.max()),
              "rotb_nonstrict_mean": pop.rotatable_bonds_nonstrict.mean(),
              "rotb_nonstrict_max": int(pop.rotatable_bonds_nonstrict.max()),
              "ring_count_mean": pop.ring_count.mean(),
              "aromatic_ring_mean": pop.aromatic_ring_count.mean(),
              "n_zero_rotb": int((pop.rotatable_bonds == 0).sum())})
    flex_rows.append(r)
flex = pd.DataFrame(flex_rows)
flex.to_csv(OUT / "section5_flexibility_summary.csv", index=False)
bin_cols = [f"n_{l}" for _, _, l in BINS]
pct_cols = [f"pct_{l}" for _, _, l in BINS]
print("rotatable-bond bins (strict count):")
print(flex[["population", "n"] + bin_cols].to_string(index=False))
print("\nas percentages:")
print(flex[["population", "n"] + pct_cols].round(1).to_string(index=False))
print("\ndistribution statistics:")
print(flex[["population", "rotb_mean", "rotb_median", "rotb_p90", "rotb_p95", "rotb_p99",
            "rotb_max", "rotb_nonstrict_mean", "rotb_nonstrict_max", "n_zero_rotb",
            "ring_count_mean", "aromatic_ring_mean"]].round(2).to_string(index=False))

rotatable-bond bins (strict count):
    population    n  n_0-2 (rigid)  n_3-5  n_6-10  n_>10 (floppy)
   train (all) 4905            624   3336     925              20
   blind (all)  750             56    520     173               1
     both sets 5655            680   3856    1098              21
train / CYP1A2 1412            216    969     217              10
train / CYP2C9 1285            160    900     223               2
train / CYP2D6 1493            164    945     374              10
train / CYP3A4 2335            289   1641     398               7

as percentages:
    population    n  pct_0-2 (rigid)  pct_3-5  pct_6-10  pct_>10 (floppy)
   train (all) 4905             12.7     68.0      18.9               0.4
   blind (all)  750              7.5     69.3      23.1               0.1
     both sets 5655             12.0     68.2      19.4               0.4
train / CYP1A2 1412             15.3     68.6      15.4               0.7
train / CYP2C9 1285             12.5     70.0    

In [18]:
# ---- what the distribution implies for a conformer budget.
# A rotatable bond with three staggered minima gives 3**k raw conformers before pruning. RDKit's
# own ETKDG guidance (and common practice) is a flat cap, so both are costed: the exponential
# ideal, and the capped reality. No conformer is generated -- these are counts, not structures.
def conformers_needed(k, cap):
    return int(min(cap, 3 ** min(k, 20)))


budget = []
for name, pop in [("both sets", AUD), ("train (all)", TR), ("blind (all)", BL)]:
    for cap in (1, 10, 50, 100, 300):
        tot = sum(conformers_needed(int(k), cap) for k in pop.rotatable_bonds)
        budget.append({"population": name, "n_compounds": len(pop), "cap_per_compound": cap,
                       "total_conformers": tot,
                       "mean_conformers_per_compound": tot / len(pop)})
budget = pd.DataFrame(budget)
budget.to_csv(OUT / "section5_conformer_budget.csv", index=False)
print("conformer count under a flat per-compound cap (3**rotatable_bonds, capped):")
print(budget.to_string(index=False))

b = flex[flex.population == "both sets"].iloc[0]
_n_above_p95 = int((AUD.rotatable_bonds > b["rotb_p95"]).sum())
_bud = budget[budget.population == "both sets"].set_index("cap_per_compound")
_cap50 = float(_bud.loc[50, "mean_conformers_per_compound"])
_cap300 = float(_bud.loc[300, "mean_conformers_per_compound"])
rigid = b["pct_0-2 (rigid)"]
mid = b["pct_3-5"]
floppy = b["pct_>10 (floppy)"]
print(f"""
READ FOR THE PROTOCOL (both sets, {int(b['n'])} compounds)
  {rigid:.1f}% sit in the rigid 0-2 bin and {mid:.1f}% in the 3-5 bin -- so
  {rigid + mid:.1f}% of all compounds have 5 or fewer rotatable bonds.
  Only {floppy:.1f}% ({int(b['n_>10 (floppy)'])} compounds) exceed 10.
  Median {b['rotb_median']:.0f}, p99 {b['rotb_p99']:.0f}, max {int(b['rotb_max'])}.

  A protocol sized for the 95th percentile ({b['rotb_p95']:.0f} rotatable bonds) leaves only
  {_n_above_p95} compounds ({pct(_n_above_p95, len(AUD)):.1f}%) above its cap, and those can carry
  a larger cap individually without moving the total materially: raising the cap from 50 to 300
  moves the mean conformers per compound from {_cap50:.1f} to {_cap300:.1f}, i.e. a
  {_cap300 / _cap50:.1f}x budget increase to serve {pct(_n_above_p95, len(AUD)):.1f}% of the set.
""")

conformer count under a flat per-compound cap (3**rotatable_bonds, capped):
 population  n_compounds  cap_per_compound  total_conformers  mean_conformers_per_compound
  both sets         5655                 1              5655                      1.000000
  both sets         5655                10             54842                      9.697966
  both sets         5655                50            228036                     40.324668
  both sets         5655               100            391654                     69.258002
  both sets         5655               300            787912                    139.330150
train (all)         4905                 1              4905                      1.000000
train (all)         4905                10             47508                      9.685627
train (all)         4905                50            195932                     39.945362
train (all)         4905               100            335207                     68.339857
train (all)   

---

## Section 6 — what is already in the feature sets: the redundancy check

If 3D content is already present in either tabular feature set, part of the planned 3D block is
redundant, and that has to be known before anything is generated.

Two things are established, both **from the saved feature files and whatever generated them**, not
from prose or from a notebook's markdown:

- whether `mordred_pca`'s source descriptor block contains any 3D descriptors. Mordred computes
  its 3D descriptors only when handed a conformer, so the question is whether any were requested.
- the full column list of `ecfp4_narrow`'s 9 non-fingerprint descriptors.

The configs are resolved through `scripts/run_5x5_cv_comparison.py`'s own `FEATURE_FILES` mapping,
read out of the script rather than assumed, since that is the registry every model in notebooks
`05`–`37` actually loads from.

In [19]:
# ---- resolve the feature files through the registry the models themselves use.
SCRIPT = REPO_ROOT / "scripts" / "run_5x5_cv_comparison.py"
src = SCRIPT.read_text()
block = src[src.index("FEATURE_FILES = {"):]
block = block[:block.index("}") + 1]
print(f"FEATURE_FILES, read out of {SCRIPT.relative_to(REPO_ROOT)}:")
print(block)
import ast as _ast
FEATURE_FILES = _ast.literal_eval(block.split("=", 1)[1].strip())
assert FEATURE_FILES["ecfp4_narrow"] == ECFP4_NARROW.name, FEATURE_FILES
assert FEATURE_FILES["mordred_pca"] == MORDRED_PCA.name, FEATURE_FILES
print("confirmed: the paths this section reads are the ones the models load.")

# ---- ecfp4_narrow: the full non-fingerprint column list.
ecfp_head = pd.read_csv(ECFP4_NARROW, nrows=1)
ID_COLS = ["Molecule_Name", "inchikey", "split"]
fp_cols = [c for c in ecfp_head.columns if c.startswith("ecfp4_")]
desc_cols = [c for c in ecfp_head.columns if c not in ID_COLS and not c.startswith("ecfp4_")]
print(f"\n{ECFP4_NARROW.name}: {len(ecfp_head.columns)} columns = "
      f"{len(ID_COLS)} id + {len(fp_cols)} fingerprint bits + {len(desc_cols)} descriptors")
assert len(desc_cols) == 9, (len(desc_cols), desc_cols)

# What each one is, taken from src/features.py's isoform_structural_descriptors (which generated
# them) -- and the dimensionality each one actually requires.
DESC_MEANING = {
    "mol_wt": ("Descriptors.MolWt", "0D (formula only)", "no"),
    "hba": ("rdMolDescriptors.CalcNumHBA", "2D (topology)", "no"),
    "hbd": ("rdMolDescriptors.CalcNumHBD", "2D (topology)", "no"),
    "ring_count": ("rdMolDescriptors.CalcNumRings", "2D (topology)", "no"),
    "stereocenter_count": ("rdMolDescriptors.CalcNumAtomStereoCenters", "2D (graph + parity)",
                           "no -- a COUNT of potential centres, see note"),
    "formal_charge": ("Chem.GetFormalCharge", "0D", "no"),
    "logp": ("Crippen.MolLogP", "2D (atom-contribution sum)", "no"),
    "logd_proxy": ("Crippen.MolLogP (same value as logp)", "2D", "no"),
    "vsa_acc_proxy": ("Descriptors.PEOE_VSA1 + PEOE_VSA2", "2D (topological VSA)", "no"),
}
ecfp_desc = pd.DataFrame([
    {"column": c, "rdkit_call": DESC_MEANING[c][0], "dimensionality": DESC_MEANING[c][1],
     "requires_a_conformer": DESC_MEANING[c][2]} for c in desc_cols])
ecfp_desc.to_csv(OUT / "section6_ecfp4_narrow_descriptors.csv", index=False)
print(f"\nthe {len(desc_cols)} non-fingerprint descriptors in ecfp4_narrow, in file order:")
print(ecfp_desc.to_string(index=False))

# logp and logd_proxy are the same number by construction -- confirmed on the real file, not
# assumed from the docstring.
full_ecfp = pd.read_csv(ECFP4_NARROW, usecols=["logp", "logd_proxy"])
same = bool(np.allclose(full_ecfp.logp, full_ecfp.logd_proxy, atol=0, rtol=0))
print(f"\nlogp and logd_proxy identical on all {len(full_ecfp)} rows: {same}  "
      f"(max abs diff {np.abs(full_ecfp.logp - full_ecfp.logd_proxy).max():.3e})")
assert same, "logp and logd_proxy differ -- src/features.py's docstring says they should not"
print("  -> so ecfp4_narrow carries 8 distinct values, not 9: logd_proxy is a duplicate column.")

FEATURE_FILES, read out of scripts/run_5x5_cv_comparison.py:
FEATURE_FILES = {
    "ecfp4_narrow": "tabular_baseline_features.csv",
    "chemeleon": "chemeleon_embeddings.npy",
    "mordred_pca": "tabular_mordred_pca.csv",
}
confirmed: the paths this section reads are the ones the models load.

tabular_baseline_features.csv: 2060 columns = 3 id + 2048 fingerprint bits + 9 descriptors

the 9 non-fingerprint descriptors in ecfp4_narrow, in file order:
            column                                rdkit_call             dimensionality                         requires_a_conformer
            mol_wt                         Descriptors.MolWt          0D (formula only)                                           no
               hba               rdMolDescriptors.CalcNumHBA              2D (topology)                                           no
               hbd               rdMolDescriptors.CalcNumHBD              2D (topology)                                           no
        ring_c

In [20]:
# ---- mordred_pca: is there any 3D descriptor in the source block?
# The saved PCA file holds components, not named descriptors, so the source block is recovered from
# outputs/mordred_pca/descriptor_nan_report.csv -- the per-descriptor NaN report written by the
# notebook that built it, which names every source descriptor.
nan_report = pd.read_csv(MORDRED_NAN_REPORT)
saved_names = set(nan_report.descriptor)
print(f"{MORDRED_NAN_REPORT.relative_to(REPO_ROOT)}: {len(saved_names)} source descriptor names")

mp_head = pd.read_csv(MORDRED_PCA, nrows=1)
mp_components = [c for c in mp_head.columns if c.startswith("mordred_pca_")]
print(f"{MORDRED_PCA.name}: {len(mp_head.columns)} columns = {len(ID_COLS)} id + "
      f"{len(mp_components)} PCA components")

from mordred import Calculator
from mordred import descriptors as mordred_descriptors

names_2d = {str(d) for d in Calculator(mordred_descriptors, ignore_3D=True).descriptors}
names_all = {str(d) for d in Calculator(mordred_descriptors, ignore_3D=False).descriptors}
only_3d = names_all - names_2d
print(f"\nMordred (installed version) exposes {len(names_2d)} descriptors with ignore_3D=True and "
      f"{len(names_all)} with ignore_3D=False")
print(f"  -> {len(only_3d)} descriptors are 3D-exclusive, i.e. require a conformer")

overlap = sorted(saved_names & only_3d)
print(f"\n3D-exclusive Mordred descriptors present in the saved source block: {len(overlap)}")
print(f"  {overlap if overlap else 'NONE'}")
print(f"saved block == the ignore_3D=True set exactly: {saved_names == names_2d}")
print(f"  saved-but-not-in-2D-set: {len(saved_names - names_2d)}   "
      f"2D-set-but-not-saved: {len(names_2d - saved_names)}")

# And the generator itself, read out of src/features.py rather than remembered.
feat_src = (REPO_ROOT / "src" / "features.py").read_text()
gen_line = [l.strip() for l in feat_src.split("\n") if "Calculator(" in l and "mordred" in l]
print(f"\nthe call in src/features.py that generated it: {gen_line}")
assert any("ignore_3D=True" in l for l in gen_line), (
    "src/features.py's Mordred call does not set ignore_3D=True -- this section's conclusion "
    "must be rechecked against the saved file alone")

mordred_rows = [
    {"check": "source descriptor names recovered from the saved NaN report",
     "value": len(saved_names)},
    {"check": "PCA components in the saved feature file", "value": len(mp_components)},
    {"check": "Mordred descriptors available with ignore_3D=True", "value": len(names_2d)},
    {"check": "Mordred descriptors available with ignore_3D=False", "value": len(names_all)},
    {"check": "Mordred descriptors that are 3D-exclusive", "value": len(only_3d)},
    {"check": "3D-exclusive descriptors found in the saved block", "value": len(overlap)},
    {"check": "saved block identical to the ignore_3D=True set",
     "value": int(saved_names == names_2d)},
    {"check": "all-NaN (structurally undefined) descriptors in the saved block",
     "value": int((nan_report.nan_rate == 1.0).sum())},
]
pd.DataFrame(mordred_rows).to_csv(OUT / "section6_mordred_3d_check.csv", index=False)
print("\nsummary:")
print(pd.DataFrame(mordred_rows).to_string(index=False))

# The named families that are MISSING, since that is what a 3D block would actually be adding.
fams = {}
for n in sorted(only_3d):
    key = ("WHIM / MoRSE (Mor*)" if n.startswith("Mor") else
           "CPSA charged-partial-surface-area" if n[:4] in {"DPSA", "FNSA", "FPSA", "PNSA", "PPSA",
                                                            "WNSA", "WPSA"} or n in
           {"RASA", "RNCS", "RPCS", "RPSA", "TASA", "TPSA"} else
           "gravitational index (GRAV*)" if n.startswith("GRAV") else
           "geometrical (GeomDiameter / GeomRadius / shape)" if n.startswith("Geom") else
           "principal moments of inertia (MOMI)" if n.startswith("MOMI") else
           "plane of best fit (PBF)" if n == "PBF" else "other")
    fams.setdefault(key, []).append(n)
print("\nthe 3D descriptor families Mordred COULD compute and this project does not have:")
for k, v in sorted(fams.items(), key=lambda kv: -len(kv[1])):
    print(f"  {len(v):4d}  {k:48s} e.g. {', '.join(v[:4])}")
pd.DataFrame([{"family": k, "n_descriptors": len(v), "names": "|".join(v)}
              for k, v in sorted(fams.items(), key=lambda kv: -len(kv[1]))]).to_csv(
    OUT / "section6_missing_3d_families.csv", index=False)
print("\nNOTE on one name that reads confusingly: Mordred files `TPSA` under its CPSA (3D) module,")
print("so `TPSA` is absent from the saved block -- but ordinary topological polar surface area IS")
print("present, under Mordred's 2D names `TopoPSA` and `TopoPSA(NO)`. Both confirmed present:")
print(f"  TopoPSA in saved block: {'TopoPSA' in saved_names}   "
      f"TopoPSA(NO): {'TopoPSA(NO)' in saved_names}   TPSA: {'TPSA' in saved_names}")

outputs/mordred_pca/descriptor_nan_report.csv: 1613 source descriptor names
tabular_mordred_pca.csv: 135 columns = 3 id + 132 PCA components



Mordred (installed version) exposes 1613 descriptors with ignore_3D=True and 1826 with ignore_3D=False
  -> 213 descriptors are 3D-exclusive, i.e. require a conformer

3D-exclusive Mordred descriptors present in the saved source block: 0
  NONE
saved block == the ignore_3D=True set exactly: True
  saved-but-not-in-2D-set: 0   2D-set-but-not-saved: 0

the call in src/features.py that generated it: ['calc = Calculator(mordred_descriptors, ignore_3D=True)']

summary:
                                                          check  value
    source descriptor names recovered from the saved NaN report   1613
                       PCA components in the saved feature file    132
              Mordred descriptors available with ignore_3D=True   1613
             Mordred descriptors available with ignore_3D=False   1826
                      Mordred descriptors that are 3D-exclusive    213
              3D-exclusive descriptors found in the saved block      0
                saved block ident

In [21]:
# ---- the two other feature sets in the registry, for completeness.
print("=" * 84)
print("SECTION 6 VERDICT")
print("=" * 84)
chemeleon_idx = REPO_ROOT / "data" / "processed" / "chemeleon_embeddings_index.csv"
chemeleon_npy = REPO_ROOT / "data" / "processed" / "chemeleon_embeddings.npy"
ch_shape = np.load(chemeleon_npy, mmap_mode="r").shape
print(f"  chemeleon       {ch_shape[1]}-dim learned embedding over {ch_shape[0]} compounds "
      f"({chemeleon_npy.name})")
print(f"  ecfp4_narrow    {len(fp_cols)} ECFP4 bits + {len(desc_cols)} descriptors "
      f"({len(desc_cols) - 1} distinct)")
print(f"  mordred_pca     {len(mp_components)} PCA components over {len(saved_names)} "
      f"2D-only Mordred descriptors")
print()
rows = [
    {"feature_set": "ecfp4_narrow", "contains_3D_content": "NO",
     "evidence": f"all {len(desc_cols)} non-fingerprint descriptors are 0D/2D RDKit calls; "
                 "none takes a conformer. ECFP4 is a topological fingerprint."},
    {"feature_set": "mordred_pca", "contains_3D_content": "NO",
     "evidence": f"saved source block is exactly the {len(names_2d)}-descriptor ignore_3D=True "
                 f"set; 0 of Mordred's {len(only_3d)} 3D-exclusive descriptors present."},
    {"feature_set": "chemeleon", "contains_3D_content": "NO",
     "evidence": "a graph-based learned embedding over 2D molecular graphs (chemprop/CheMeleon); "
                 "no coordinates enter it."},
]
verdict6 = pd.DataFrame(rows)
verdict6.to_csv(OUT / "section6_feature_redundancy.csv", index=False)
print(verdict6.to_string(index=False))
print()
print("  NOTHING PLANNED AS A 3D BLOCK IS ALREADY PRESENT. No conformer-dependent descriptor")
print("  exists anywhere in this project's three feature sets, so a 3D block would be genuinely")
print("  new information rather than a re-encoding.")
print()
print("  ONE PARTIAL EXCEPTION, worth stating because it is easy to misread as 3D content:")
print(f"  ecfp4_narrow carries `stereocenter_count`, built from CalcNumAtomStereoCenters. Part 0")
print("  established by direct test that this call counts SPECIFIED AND UNSPECIFIED centres")
print("  together, so the column is a count of potential chiral centres -- it says a centre is")
print("  there, never which configuration it has, and carries no geometry. It overlaps the")
print("  'does this compound have a stereocentre' part of a chirality block and none of the rest.")
print()
print("  Also note `logd_proxy` duplicates `logp` exactly (checked above), so one of")
print("  ecfp4_narrow's nine descriptor columns carries no information the other eight do not.")

SECTION 6 VERDICT
  chemeleon       2048-dim learned embedding over 5655 compounds (chemeleon_embeddings.npy)
  ecfp4_narrow    2048 ECFP4 bits + 9 descriptors (8 distinct)
  mordred_pca     132 PCA components over 1613 2D-only Mordred descriptors

 feature_set contains_3D_content                                                                                                                   evidence
ecfp4_narrow                  NO       all 9 non-fingerprint descriptors are 0D/2D RDKit calls; none takes a conformer. ECFP4 is a topological fingerprint.
 mordred_pca                  NO saved source block is exactly the 1613-descriptor ignore_3D=True set; 0 of Mordred's 213 3D-exclusive descriptors present.
   chemeleon                  NO                    a graph-based learned embedding over 2D molecular graphs (chemprop/CheMeleon); no coordinates enter it.

  NOTHING PLANNED AS A 3D BLOCK IS ALREADY PRESENT. No conformer-dependent descriptor
  exists anywhere in this project's thre

---

## Figures

House style, each with a caption stating the finding rather than describing the axes.

In [22]:
plt.rcParams.update({"font.size": 10, "figure.dpi": 110, "savefig.dpi": 150,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
                     "legend.frameon": False})
GREY, ACCENT, MID, GOOD = "#8C8C8C", "#B0413E", "#4C4C4C", "#2C7A4B"
ISO_COLOURS = {"CYP1A2": "#4C72B0", "CYP2C9": "#DD8452", "CYP2D6": "#B0413E", "CYP3A4": "#2C7A4B"}


def finish(fig, path):
    fig.savefig(path, bbox_inches="tight", dpi=150)
    plt.close(fig)
    print(f"wrote {path.relative_to(REPO_ROOT)}")

In [23]:
# ---- FIGURE 1: heavy-atom distribution per isoform, with the tail made visible.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.4),
                               gridspec_kw={"width_ratios": [1.3, 1], "wspace": 0.28})

bins = np.arange(AUD.heavy_atoms.min() - 0.5, AUD.heavy_atoms.max() + 1.5, 1)
for iso in ISOFORMS:
    pop = TR[TR[f"has_{iso}"]]
    ax1.hist(pop.heavy_atoms, bins=bins, histtype="step", lw=1.5, density=True,
             color=ISO_COLOURS[iso], label=f"{iso} (n={len(pop)})")
ax1.hist(BL.heavy_atoms, bins=bins, histtype="stepfilled", density=True, color=GREY, alpha=0.35,
         label=f"blind (n={len(BL)})", zorder=0)
ax1.set_xlim(5, 45)
ax1.set_xlabel("heavy atoms")
ax1.set_ylabel("density")
ax1.set_title("the bulk: four isoforms and the blind set are near-identical", fontsize=10)
ax1.legend(fontsize=8)

# The tail, on a log count axis so single compounds remain visible.
for iso in ISOFORMS:
    pop = TR[TR[f"has_{iso}"]]
    cnt, edges = np.histogram(pop.heavy_atoms, bins=bins)
    mask = cnt > 0
    ax2.step(edges[:-1][mask] + 0.5, cnt[mask], where="mid", lw=1.4, color=ISO_COLOURS[iso],
             label=iso)
cnt, edges = np.histogram(BL.heavy_atoms, bins=bins)
mask = cnt > 0
ax2.step(edges[:-1][mask] + 0.5, cnt[mask], where="mid", lw=1.4, color=GREY, ls="--", label="blind")
TAIL_FROM = 32
ax2.set_yscale("log")
ax2.set_xlim(TAIL_FROM, AUD.heavy_atoms.max() + 2)
ax2.set_ylim(0.6, max(1.0, float(np.histogram(AUD.heavy_atoms, bins=bins)[0]
                                 [bins[:-1] >= TAIL_FROM].max())) * 3)
ax2.set_xlabel("heavy atoms")
ax2.set_ylabel("compounds (log scale)")
ax2.set_title(f"the tail: every compound above {TAIL_FROM} heavy atoms", fontsize=10)
ax2.legend(fontsize=8, ncol=2, loc="upper right")
for t, style in [(50, ":"), (60, "-.")]:
    ax2.axvline(t, color=MID, lw=0.9, ls=style)
    ax2.text(t + 0.5, 0.75, f"{t} HA", fontsize=7.5, color=MID, va="bottom")
# Name the three compounds the tail actually consists of, since there are few enough to name.
for r in AUD.nlargest(3, "heavy_atoms").itertuples():
    ax2.annotate(r.Molecule_Name, (r.heavy_atoms, 1.0), textcoords="offset points",
                 xytext=(0, 12), ha="center", fontsize=6.5, color=ACCENT, rotation=90)
_n50 = int((AUD.heavy_atoms > 50).sum())
_p99 = float(np.percentile(AUD.heavy_atoms, 99))
fig.suptitle(f"The set has almost no size tail: p99 = {_p99:.0f} heavy atoms and only {_n50} of "
             f"{len(AUD)} compounds exceed 50",
             fontsize=11, fontweight="bold", x=0.01, ha="left")
fig.subplots_adjust(top=0.84)
finish(fig, FIG_OUT / "heavy_atom_distribution.png")

wrote outputs/41_dataset_audit_3d/figures/heavy_atom_distribution.png


**Figure 1 — `heavy_atom_distribution.png`.** Heavy-atom count per compound. *Left:* the bulk of
the distribution, each isoform's labelled subset as an outline and the blind set shaded, all
density-normalised so the four unequal populations are comparable — they lie almost on top of each
other, so no isoform brings a distinctively larger or smaller chemotype. *Right:* the tail alone, from 32
heavy atoms up, as raw counts on a log axis — the only way a tail of three compounds stays
visible — with the three largest named and the 50- and 60-heavy-atom marks drawn in. The finding is the
**absence** of what the section was built to measure: this is a tightly drug-like library with a
99th percentile around 33 heavy atoms, not a set with a cost-dominating tail.

In [24]:
# ---- FIGURE 2: rotatable-bond distribution, which sizes the conformer protocol.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.3),
                               gridspec_kw={"width_ratios": [1.25, 1], "wspace": 0.3})

rb_bins = np.arange(-0.5, max(AUD.rotatable_bonds.max(), 14) + 1.5, 1)
for iso in ISOFORMS:
    pop = TR[TR[f"has_{iso}"]]
    ax1.hist(pop.rotatable_bonds, bins=rb_bins, histtype="step", lw=1.5, density=True,
             color=ISO_COLOURS[iso], label=iso)
ax1.hist(BL.rotatable_bonds, bins=rb_bins, histtype="stepfilled", density=True, color=GREY,
         alpha=0.35, label="blind", zorder=0)
ax1.set_xlim(-0.5, 14.5)
ax1.set_xlabel("rotatable bonds (RDKit strict)")
ax1.set_ylabel("density")
ax1.set_title("distribution", fontsize=10)
ax1.legend(fontsize=8, ncol=2)

labels = [l for _, _, l in BINS]
x = np.arange(len(labels))
w = 0.38
tr_pct = [flex[flex.population == "train (all)"].iloc[0][f"pct_{l}"] for l in labels]
bl_pct = [flex[flex.population == "blind (all)"].iloc[0][f"pct_{l}"] for l in labels]
ax2.bar(x - w / 2, tr_pct, w, color=MID, label="train")
ax2.bar(x + w / 2, bl_pct, w, color=GREY, label="blind")
def _fmt(v):
    # A bin holding real compounds must never print as "0%": the >10 bin is 0.4% and 0.1%.
    return f"{v:.0f}%" if v >= 1 else f"{v:.1f}%"


for xi, (a, b) in enumerate(zip(tr_pct, bl_pct)):
    ax2.text(xi - w / 2, a, _fmt(a), ha="center", va="bottom", fontsize=8)
    ax2.text(xi + w / 2, b, _fmt(b), ha="center", va="bottom", fontsize=8)
ax2.set_xticks(x)
ax2.set_xticklabels(labels, fontsize=8.5)
ax2.set_ylabel("% of compounds")
ax2.set_title("the bins that size a conformer protocol", fontsize=10)
ax2.legend(fontsize=8)
ax2.set_ylim(0, max(tr_pct + bl_pct) * 1.18)
_b = flex[flex.population == "both sets"].iloc[0]
fig.suptitle(f"{_b['pct_0-2 (rigid)'] + _b['pct_3-5']:.0f}% of compounds have 5 or fewer rotatable "
             f"bonds; only {int(_b['n_>10 (floppy)'])} of {len(AUD)} exceed 10",
             fontsize=11, fontweight="bold", x=0.01, ha="left")
fig.subplots_adjust(top=0.85)
finish(fig, FIG_OUT / "rotatable_bond_distribution.png")

wrote outputs/41_dataset_audit_3d/figures/rotatable_bond_distribution.png


**Figure 2 — `rotatable_bond_distribution.png`.** Rotatable bonds per compound, the quantity that
multiplies every other cost in a conformer pipeline. *Left:* the distribution per isoform and for
the blind set, density-normalised. *Right:* the four bins the protocol would branch on, train
against blind. The distribution is narrow and centred on 4, with a very thin floppy tail, so a
conformer protocol can be sized on a single cap for almost the whole library and the exceptions
named individually. Train and blind agree closely enough that a protocol tuned on one transfers to
the other.

In [25]:
# ---- FIGURE 3: stereochemistry status -- the figure that answers the chirality question.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5),
                               gridspec_kw={"width_ratios": [1.45, 1], "wspace": 0.3})

order = ["train (all)", "blind (all)"] + [f"train / {i}" for i in ISOFORMS]
S = stereo_tbl.set_index("population").loc[order]
y = np.arange(len(order))
no_c = S.pct_no_centre.to_numpy()
full = S.pct_fully_specified.to_numpy()
unas = S.pct_ge1_unassigned.to_numpy()
ax1.barh(y, no_c, color=GREY, label="no stereocentre at all")
ax1.barh(y, full, left=no_c, color=GOOD, label="every centre specified")
ax1.barh(y, unas, left=no_c + full, color=ACCENT, label="at least one centre UNASSIGNED")
ax1.set_yticks(y)
ax1.set_yticklabels([o.replace("train / ", "") for o in order], fontsize=9)
ax1.invert_yaxis()
ax1.set_xlabel("% of compounds")
ax1.set_xlim(0, 100)
ax1.set_title("stereochemical status of every compound", fontsize=10)
ax1.legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=3)
for yi, (a, b) in enumerate(zip(no_c, full)):
    if b >= 2.0:
        ax1.text(a + b / 2, yi, f"{b:.0f}%", ha="center", va="center", fontsize=7.5,
                 color="white", fontweight="bold")
for yi, (a, b, c) in enumerate(zip(no_c, full, unas)):
    ax1.text(a + b + c / 2, yi, f"{c:.0f}%", ha="center", va="center", fontsize=7.5,
             color="white", fontweight="bold")

# Of the compounds that HAVE a centre, what share is undetermined -- the number that bites.
share = S.pct_of_centred_compounds_unassigned.to_numpy()
cols = [ACCENT if s > 90 else MID for s in share]
ax2.barh(y, share, color=cols, height=0.62)
ax2.axvline(100, color="black", lw=1.0)
ax2.set_yticks(y)
ax2.set_yticklabels([])
ax2.invert_yaxis()
ax2.set_xlim(0, 108)
ax2.set_xlabel("% of centre-bearing compounds with >=1 unassigned centre")
ax2.set_title("100% = configuration never stated", fontsize=10)
for yi, s in enumerate(share):
    ax2.text(s + 1.5, yi, f"{s:.1f}%", va="center", fontsize=8, fontweight="bold")
_bl = stereo_tbl[stereo_tbl.population == "blind (all)"].iloc[0]
fig.suptitle(f"Only {int(_bl['n_fully_specified'])} of the {int(_bl['n'])} blind compounds have "
             f"every stereocentre specified\n"
             f"{_bl['pct_of_centred_compounds_unassigned']:.0f}% of those with a centre do not say "
             f"which enantiomer was tested",
             fontsize=11, fontweight="bold", x=0.01, ha="left")
fig.subplots_adjust(top=0.80, bottom=0.20)
finish(fig, FIG_OUT / "stereochemistry_status.png")

wrote outputs/41_dataset_audit_3d/figures/stereochemistry_status.png


**Figure 3 — `stereochemistry_status.png`.** The figure the chirality decision turns on. *Left:*
every compound in each population sorted into three mutually exclusive states — no stereocentre at
all (grey), a centre with every configuration specified (green), or at least one centre whose
configuration the data never states (red). *Right:* restricted to compounds that actually have a
centre, the share whose configuration is undetermined; 100% would mean a centre is never once
specified. The left panel alone understates the problem, because a compound with no centre needs no
configuration — it is the right panel that shows the data is almost silent wherever chirality could
matter, and most sharply so on the blind set, which is the population that gets scored. Note that
the per-isoform rows overlap: a compound counts toward every isoform it carries a label for.

In [26]:
# ---- FIGURE 4: basic nitrogen per isoform -- sizes the nitrogen-local block.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.3),
                               gridspec_kw={"width_ratios": [1.15, 1], "wspace": 0.3})

rows_o = [f"train / {i}" for i in ISOFORMS] + ["train (all)", "blind (all)"]
B = bn.set_index("population").loc[rows_o]
x = np.arange(len(rows_o))
w = 0.38
ax1.bar(x - w / 2, B.pct_ge1_strict, w, color=MID, label="strict (aliphatic, non-amide only)")
ax1.bar(x + w / 2, B.pct_ge1_loose, w, color=GREY, label="loose (aniline-type N also counted)")
for xi, (a, b) in enumerate(zip(B.pct_ge1_strict, B.pct_ge1_loose)):
    ax1.text(xi - w / 2, a, f"{a:.0f}", ha="center", va="bottom", fontsize=8)
    ax1.text(xi + w / 2, b, f"{b:.0f}", ha="center", va="bottom", fontsize=8)
for xi, lbl in enumerate(rows_o):
    if lbl == "train / CYP2D6":
        ax1.axvspan(xi - 0.5, xi + 0.5, color=ACCENT, alpha=0.08, zorder=0)
ax1.set_xticks(x)
ax1.set_xticklabels([o.replace("train / ", "").replace(" (all)", "") for o in rows_o], fontsize=8.5)
ax1.set_ylabel("% of compounds with >=1 basic nitrogen")
ax1.set_title("CYP2D6 stands apart under both definitions", fontsize=10)
ax1.legend(fontsize=8)
ax1.set_ylim(0, float(B.pct_ge1_loose.max()) * 1.32)

counts = np.array([[B.loc[r, "n_zero_strict"], B.loc[r, "n_one_strict"],
                    B.loc[r, "n_two_plus_strict"]] for r in rows_o], dtype=float)
frac = 100 * counts / counts.sum(axis=1, keepdims=True)
left = np.zeros(len(rows_o))
for j, (lab, col) in enumerate([("none", GREY), ("one", MID), ("two or more", ACCENT)]):
    ax2.barh(x, frac[:, j], left=left, color=col, label=lab)
    left += frac[:, j]
ax2.set_yticks(x)
ax2.set_yticklabels([o.replace("train / ", "").replace(" (all)", "") for o in rows_o], fontsize=8.5)
ax2.invert_yaxis()
ax2.set_xlim(0, 100)
ax2.set_xlabel("% of compounds")
ax2.set_title("how many basic nitrogens (strict)", fontsize=10)
ax2.legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=3)
_d6 = B.loc["train / CYP2D6", "pct_ge1_strict"]
_oth = float(np.mean([B.loc[f"train / {i}", "pct_ge1_strict"]
                      for i in ISOFORMS if i != "CYP2D6"]))
fig.suptitle(f"CYP2D6's compounds carry a basic nitrogen {_d6 / _oth:.1f}x as often as the other "
             f"three isoforms' ({_d6:.1f}% against {_oth:.1f}%)",
             fontsize=11, fontweight="bold", x=0.01, ha="left")
fig.subplots_adjust(top=0.85, bottom=0.20)
finish(fig, FIG_OUT / "basic_nitrogen_by_isoform.png")

wrote outputs/41_dataset_audit_3d/figures/basic_nitrogen_by_isoform.png


**Figure 4 — `basic_nitrogen_by_isoform.png`.** *Left:* the share of each population's compounds
carrying at least one basic nitrogen, under the strict definition (aliphatic, non-amide,
non-aromatic) and under a looser one that also counts aniline-type nitrogen. CYP2D6's row is
shaded. The two definitions disagree on the absolute level by roughly a factor of two, and agree
completely on the ordering — which is why both are carried rather than one being chosen. *Right:*
the full none / one / two-or-more breakdown under the strict rule, which is what sizes a
nitrogen-local descriptor block: the block is empty for the grey portion of each bar. Read with
Figure 3: CYP2D6 is the isoform where the mechanistic argument for a nitrogen-local descriptor is
strongest and also the isoform notebook `17` found carries a charge signal not explained by a
lipophilicity confound.

---

## Verdict — the four questions, answered directly

Each answer is computed from the tables above rather than restated from them, so the numbers in the
verdict cannot drift from the numbers in the sections. The verdict is also written to
`verdict.json`.

In [27]:
V = {}
tr_s = stereo_tbl[stereo_tbl.population == "train (all)"].iloc[0]
bl_s = stereo_tbl[stereo_tbl.population == "blind (all)"].iloc[0]
both_sz = size_tbl[(size_tbl.population == "both sets") & (size_tbl.quantity == "heavy_atoms")].iloc[0]
both_fl = flex[flex.population == "both sets"].iloc[0]
bn_tr = bn[bn.population == "train (all)"].iloc[0]
bn_bl = bn[bn.population == "blind (all)"].iloc[0]
d6 = bn[bn.population == "train / CYP2D6"].iloc[0]
oth = bn[bn.population.isin([f"train / {i}" for i in ISOFORMS if i != "CYP2D6"])]

print("=" * 84)
print("Q1 -- IS A CHIRALITY / 3D-SHAPE ANGLE ALIVE, OR IS THE SET EFFECTIVELY FLAT?")
print("=" * 84)
V["q1_chirality"] = {
    "answer": "SPLIT: 3D shape is alive; chirality specifically is NOT, because the data does "
              "not state configuration for almost every compound that has a centre",
    "train_pct_ge1_centre": round(float(tr_s.pct_ge1_centre), 1),
    "train_pct_fully_specified": round(float(tr_s.pct_fully_specified), 1),
    "blind_n_fully_specified": int(bl_s.n_fully_specified),
    "blind_n": int(bl_s.n),
    "blind_pct_of_centred_unassigned": round(float(bl_s.pct_of_centred_compounds_unassigned), 1),
    "double_bond_stereo_defined_train": int(tr_s.n_db_stereo_defined),
    "double_bond_stereo_defined_blind": int(bl_s.n_db_stereo_defined),
}
print(f"""
  The set is NOT flat in the geometric sense. {tr_s['pct_ge1_centre']:.1f}% of training compounds
  and {bl_s['pct_ge1_centre']:.1f}% of blind compounds carry at least one potential stereocentre,
  and {tr_s['pct_gt1_centre']:.1f}% of training compounds carry more than one -- so sp3 character
  and three-dimensional shape are genuinely present, and a conformer-derived SHAPE descriptor has
  real variation to work with.

  But CHIRALITY SPECIFICALLY IS NOT A LIVE ANGLE, and the blind set is where it dies.
    - training: {int(tr_s['n_fully_specified'])} of {int(tr_s['n'])} compounds ({tr_s['pct_fully_specified']:.1f}%) have every centre specified.
    - blind:    {int(bl_s['n_fully_specified'])} of {int(bl_s['n'])} compounds ({bl_s['pct_fully_specified']:.1f}%).
    - of blind compounds that HAVE a centre, {bl_s['pct_of_centred_compounds_unassigned']:.1f}% do not say which configuration was tested.
    - defined double-bond stereochemistry: {int(tr_s['n_db_stereo_defined'])} training and {int(bl_s['n_db_stereo_defined'])} blind compounds.

  A chirality descriptor therefore cannot be computed from the data for the great majority of the
  compounds that are scored -- it would have to invent a configuration, and an invented
  configuration is a free parameter dressed as a measurement. Section 3 confirmed this is a
  property of the released files, not of this project's curation: the raw challenge CSVs carry
  identical counts, and canonicalisation changes nothing on any compound.

  WHAT IS STILL AVAILABLE: a shape/conformer block computed on an arbitrary-but-consistent
  configuration (generate one, use it everywhere, declare it) measures sp3/ring geometry honestly
  while making no claim about enantiomer. That is a defensible descriptor. A descriptor that
  claims to separate enantiomers is not, on this data.
""")

Q1 -- IS A CHIRALITY / 3D-SHAPE ANGLE ALIVE, OR IS THE SET EFFECTIVELY FLAT?

  The set is NOT flat in the geometric sense. 50.1% of training compounds
  and 35.5% of blind compounds carry at least one potential stereocentre,
  and 15.8% of training compounds carry more than one -- so sp3 character
  and three-dimensional shape are genuinely present, and a conformer-derived SHAPE descriptor has
  real variation to work with.

  But CHIRALITY SPECIFICALLY IS NOT A LIVE ANGLE, and the blind set is where it dies.
    - training: 459 of 4905 compounds (9.4%) have every centre specified.
    - blind:    18 of 750 compounds (2.4%).
    - of blind compounds that HAVE a centre, 93.2% do not say which configuration was tested.
    - defined double-bond stereochemistry: 19 training and 0 blind compounds.

  A chirality descriptor therefore cannot be computed from the data for the great majority of the
  compounds that are scored -- it would have to invent a configuration, and an invented
  confi

In [28]:
print("=" * 84)
print("Q2 -- WHAT DOES THE SIZE TAIL IMPLY FOR A QM BUDGET?")
print("=" * 84)
big = largest.iloc[0]
ref25 = cost_tbl[(cost_tbl.scaling_exponent == 2.5)
                 & (cost_tbl.population == "both sets")].iloc[0]
V["q2_size_tail"] = {
    "answer": "There is essentially no cost-dominating tail. The library is tightly drug-like and "
              "a flat per-compound budget is close to adequate; only a named handful needs "
              "special handling.",
    "heavy_atoms_median": float(both_sz["median"]),
    "heavy_atoms_p99": float(both_sz["p99"]),
    "heavy_atoms_max": float(both_sz["max"]),
    "n_gt_50_HA_both_sets": int((AUD.heavy_atoms > 50).sum()),
    "n_gt_50_HA_blind": int((BL.heavy_atoms > 50).sum()),
    "top1pct_cost_share_exp2.5": round(float(ref25["top1pct_share"]), 4),
    "top10pct_cost_share_exp2.5": round(float(ref25["top10pct_share"]), 4),
    "largest_compound": str(big.Molecule_Name),
    "largest_compound_heavy_atoms": int(big.heavy_atoms),
    "isoform_with_largest_p99": str(iso_tail.loc[iso_tail.HA_p99.idxmax(), "isoform"]),
    "isoform_with_largest_max": str(iso_tail.loc[iso_tail.HA_max.idxmax(), "isoform"]),
}
print(f"""
  THE TAIL IS NOT THE PROBLEM THE BRIEF EXPECTED IT TO BE, and that is the finding.
    median {both_sz['median']:.0f} heavy atoms, p95 {both_sz['p95']:.0f}, p99 {both_sz['p99']:.0f}, max {both_sz['max']:.0f}.
    above 50 heavy atoms: {int((AUD.heavy_atoms > 50).sum())} of {len(AUD)} compounds, {int((BL.heavy_atoms > 50).sum())} of them in the blind set.
    above 60: {int((AUD.heavy_atoms > 60).sum())}.   above 70: {int((AUD.heavy_atoms > 70).sum())}.

  At a cost exponent of 2.5 the largest 1% of compounds carries {ref25['top1pct_share'] * 100:.1f}% of predicted total
  cost and the largest 10% carries {ref25['top10pct_share'] * 100:.1f}% -- against the 1% and 10% a perfectly flat cost
  would give. The skew is real but mild and does not change the shape of a budget; the conclusion
  is the same at exponents 2.0, 3.0 and 4.0 (see section1_cost_model.csv).

  WHICH COMPOUNDS DOMINATE: {iso_tail.loc[iso_tail.HA_p99.idxmax(), 'isoform']} carries the largest p99 and {iso_tail.loc[iso_tail.HA_max.idxmax(), 'isoform']} the single
  largest compound, but the spread in p99 across all four isoforms is only {iso_tail.HA_p99.max() - iso_tail.HA_p99.min():.1f} heavy atoms --
  no isoform needs its own budget. What dominates is a named list of about ten compounds, not an
  isoform: {big.Molecule_Name} at {int(big.heavy_atoms)} heavy atoms ({int(big.total_atoms_with_h)} with hydrogens) costs roughly
  {(int(big.heavy_atoms) / both_sz['median']) ** 2.5:.0f}x the median compound at exponent 2.5. Section 1 names all ten with SMILES.

  TWO THINGS THAT DO WARRANT ATTENTION, neither of them size:
    - hydrogens. Heavy-atom counts understate the system an optimiser sees by a factor of about
      {AUD.total_atoms_with_h.mean() / AUD.heavy_atoms.mean():.2f}: the largest compound is {int(AUD.total_atoms_with_h.max())} atoms with hydrogens, not {int(AUD.heavy_atoms.max())}.
    - convergence risk sits in a chemotype, not a size bin. The extended cyanine dye and the
      polyiodinated compounds named in Section 2 are where SCF trouble actually clusters, and
      several of them sit well below 50 heavy atoms.
""")

Q2 -- WHAT DOES THE SIZE TAIL IMPLY FOR A QM BUDGET?

  THE TAIL IS NOT THE PROBLEM THE BRIEF EXPECTED IT TO BE, and that is the finding.
    median 24 heavy atoms, p95 30, p99 34, max 65.
    above 50 heavy atoms: 3 of 5655 compounds, 0 of them in the blind set.
    above 60: 1.   above 70: 0.

  At a cost exponent of 2.5 the largest 1% of compounds carries 3.0% of predicted total
  cost and the largest 10% carries 18.0% -- against the 1% and 10% a perfectly flat cost
  would give. The skew is real but mild and does not change the shape of a budget; the conclusion
  is the same at exponents 2.0, 3.0 and 4.0 (see section1_cost_model.csv).

  WHICH COMPOUNDS DOMINATE: CYP2C9 carries the largest p99 and CYP2D6 the single
  largest compound, but the spread in p99 across all four isoforms is only 2.1 heavy atoms --
  no isoform needs its own budget. What dominates is a named list of about ten compounds, not an
  isoform: OCNT-2328942 at 65 heavy atoms (119 with hydrogens) costs roughly
  1

In [29]:
print("=" * 84)
print("Q3 -- DOES THE BASIC-NITROGEN BLOCK COVER ENOUGH COMPOUNDS TO BE WORTH COMPUTING?")
print("=" * 84)
V["q3_basic_nitrogen"] = {
    "answer": "NOT as a whole-library block -- it is empty for 4 compounds in 5. YES as a "
              "CYP2D6-specific block, where coverage is ~3x higher and the mechanism is "
              "independently supported.",
    "train_pct_ge1_strict": round(float(bn_tr.pct_ge1_strict), 1),
    "train_pct_ge1_loose": round(float(bn_tr.pct_ge1_loose), 1),
    "blind_pct_ge1_strict": round(float(bn_bl.pct_ge1_strict), 1),
    "blind_pct_ge1_loose": round(float(bn_bl.pct_ge1_loose), 1),
    "cyp2d6_pct_ge1_strict": round(float(d6.pct_ge1_strict), 1),
    "other_isoforms_pct_ge1_strict_range": [round(float(oth.pct_ge1_strict.min()), 1),
                                            round(float(oth.pct_ge1_strict.max()), 1)],
    "cyp2d6_enrichment_strict": round(float(d6.pct_ge1_strict / oth.pct_ge1_strict.mean()), 2),
    "cyp2d6_enrichment_loose": round(float(d6.pct_ge1_loose / oth.pct_ge1_loose.mean()), 2),
    "cyp2d6_n_covered_strict": int(d6.n_ge1_strict),
    "smarts_reference_validation": f"{int(ref.PASS.sum())}/{len(ref)} PASS",
}
print(f"""
  AS A WHOLE-LIBRARY BLOCK: NO, it would be mostly empty.
    training {bn_tr['pct_ge1_strict']:.1f}% of compounds carry a basic nitrogen under the strict definition
    ({bn_tr['pct_ge1_loose']:.1f}% under the loose one); blind {bn_bl['pct_ge1_strict']:.1f}% (loose {bn_bl['pct_ge1_loose']:.1f}%). So on the blind set the
    strict block is undefined for {int(bn_bl['n_zero_strict'])} of {int(bn_bl['n'])} compounds. Every nitrogen-local feature
    would need an imputed value for most of the library, and an imputed value in a
    mechanistic descriptor is a constant wearing a mechanism's name.

  AS A CYP2D6-SPECIFIC BLOCK: YES, and this is the strongest positive result in the audit.
    CYP2D6  {d6['pct_ge1_strict']:.1f}% ({int(d6['n_ge1_strict'])} of {int(d6['n'])} compounds)      others  {oth.pct_ge1_strict.min():.1f}-{oth.pct_ge1_strict.max():.1f}%
    enrichment {d6['pct_ge1_strict'] / oth.pct_ge1_strict.mean():.2f}x strict, {d6['pct_ge1_loose'] / oth.pct_ge1_loose.mean():.2f}x loose -- the ordering survives the one
    definitional choice this section had to make, which is why both were carried.

  The enrichment is a real, structural property of which compounds were put on the CYP2D6 assay,
  measured here for the first time in this project, and it lines up with notebook 17's independent
  finding that CYP2D6 is the one isoform whose charge-pIC50 correlation survives a partial
  correlation against Crippen logP (+0.1691 raw, +0.1772 partial, while CYP1A2 and CYP2C9 are
  substantially logP-confounded and CYP3A4 collapses to -0.0029).

  CAVEAT CARRIED, NOT BURIED: notebook 18 already built a charge-augmented Chemprop variant on
  exactly this reasoning and found a clean TIE on all four isoforms, with the charge arm's
  predictions pool-typical rather than decorrelated. So prior coverage being adequate is NOT
  evidence that a richer nitrogen-local block will help -- the one time this mechanism was given
  to a model here, it did not reach the output in a way that mattered. The honest statement is
  that the block is well enough populated on CYP2D6 to be worth computing, and that whether it
  helps is a separate question with a discouraging precedent.
""")

Q3 -- DOES THE BASIC-NITROGEN BLOCK COVER ENOUGH COMPOUNDS TO BE WORTH COMPUTING?

  AS A WHOLE-LIBRARY BLOCK: NO, it would be mostly empty.
    training 18.9% of compounds carry a basic nitrogen under the strict definition
    (36.2% under the loose one); blind 11.6% (loose 19.5%). So on the blind set the
    strict block is undefined for 663 of 750 compounds. Every nitrogen-local feature
    would need an imputed value for most of the library, and an imputed value in a
    mechanistic descriptor is a constant wearing a mechanism's name.

  AS A CYP2D6-SPECIFIC BLOCK: YES, and this is the strongest positive result in the audit.
    CYP2D6  37.4% (558 of 1493 compounds)      others  12.7-13.8%
    enrichment 2.84x strict, 1.83x loose -- the ordering survives the one
    definitional choice this section had to make, which is why both were carried.

  The enrichment is a real, structural property of which compounds were put on the CYP2D6 assay,
  measured here for the first time in this 

In [30]:
print("=" * 84)
print("Q4 -- IS ANY OF THE PLANNED 3D CONTENT ALREADY IN THE EXISTING FEATURE SETS?")
print("=" * 84)
V["q4_redundancy"] = {
    "answer": "NO. No conformer-dependent descriptor exists in any of the three feature sets, so "
              "a 3D block would be new information rather than a re-encoding.",
    "mordred_source_descriptors": len(saved_names),
    "mordred_3d_exclusive_available": len(only_3d),
    "mordred_3d_exclusive_present": len(overlap),
    "mordred_block_is_exactly_the_2d_set": bool(saved_names == names_2d),
    "ecfp4_narrow_descriptor_columns": desc_cols,
    "ecfp4_narrow_distinct_values": len(desc_cols) - 1,
    "ecfp4_narrow_3d_columns": 0,
    "partial_exception": "ecfp4_narrow's stereocenter_count is a count of POTENTIAL centres "
                         "(CalcNumAtomStereoCenters merges specified and unspecified, confirmed "
                         "by direct test in Part 0) -- it carries no configuration and no geometry",
    "duplicate_column": "logd_proxy is bit-identical to logp on all rows",
}
print(f"""
  NO -- and this was checked against the saved files, not against any notebook's description.

  mordred_pca: the source descriptor block is EXACTLY Mordred's {len(names_2d)}-descriptor
    ignore_3D=True set (set equality, not just matching counts), and contains
    {len(overlap)} of Mordred's {len(only_3d)} 3D-exclusive descriptors. The
    {len(mp_components)} saved PCA components are a rotation of 2D descriptors only. The
    generating call in src/features.py sets ignore_3D=True, which is consistent, but the
    conclusion rests on the saved block itself.

  ecfp4_narrow: all {len(desc_cols)} non-fingerprint descriptors are 0D or 2D RDKit calls, none of
    which takes a conformer -- {', '.join(desc_cols)}. ECFP4 itself is topological.
    Two incidental findings: logd_proxy is bit-identical to logp, so the set carries
    {len(desc_cols) - 1} distinct values rather than {len(desc_cols)}; and stereocenter_count is a
    count of POTENTIAL centres, so it overlaps only the 'does a centre exist' part of a chirality
    block and none of the geometry.

  chemeleon: a learned embedding over 2D molecular graphs. No coordinates enter it.

  So the whole of a planned 3D block is new. What Mordred alone could add, if a conformer existed,
  is {len(only_3d)} descriptors across the families named in Section 6 -- WHIM/MoRSE, charged
  partial surface area, gravitational index, geometrical, principal moments of inertia and plane of
  best fit. None of it is currently present in any form.
""")

Q4 -- IS ANY OF THE PLANNED 3D CONTENT ALREADY IN THE EXISTING FEATURE SETS?

  NO -- and this was checked against the saved files, not against any notebook's description.

  mordred_pca: the source descriptor block is EXACTLY Mordred's 1613-descriptor
    ignore_3D=True set (set equality, not just matching counts), and contains
    0 of Mordred's 213 3D-exclusive descriptors. The
    132 saved PCA components are a rotation of 2D descriptors only. The
    generating call in src/features.py sets ignore_3D=True, which is consistent, but the
    conclusion rests on the saved block itself.

  ecfp4_narrow: all 9 non-fingerprint descriptors are 0D or 2D RDKit calls, none of
    which takes a conformer -- mol_wt, hba, hbd, ring_count, stereocenter_count, formal_charge, logp, logd_proxy, vsa_acc_proxy. ECFP4 itself is topological.
    Two incidental findings: logd_proxy is bit-identical to logp, so the set carries
    8 distinct values rather than 9; and stereocenter_count is a
    count of P

In [31]:
# ---- one combined summary table, plus the verdict as JSON.
summary = pd.DataFrame([
    {"section": "1 size", "headline":
        f"median {both_sz['median']:.0f} HA, p99 {both_sz['p99']:.0f}, max {both_sz['max']:.0f}; "
        f"{int((AUD.heavy_atoms > 50).sum())} of {len(AUD)} compounds above 50 HA",
     "file": "section1_size_summary.csv"},
    {"section": "2 elements", "headline":
        f"{len(set(el_tbl[el_tbl.population == 'both sets'].element))} elements, "
        f"{len(offenders)} compounds outside C/N/O/S/P/F/Cl/Br/I, {len(rad)} radicals, "
        f"{int((AUD.formal_charge != 0).sum())} non-neutral as written",
     "file": "section2_element_inventory.csv"},
    {"section": "3 stereo", "headline":
        f"{tr_s['pct_ge1_centre']:.0f}% of train / {bl_s['pct_ge1_centre']:.0f}% of blind have a "
        f"centre, but only {int(bl_s['n_fully_specified'])} of {int(bl_s['n'])} blind compounds "
        f"have every centre specified; canonicalisation preserves stereo exactly",
     "file": "section3_stereo_summary.csv"},
    {"section": "4 basic N", "headline":
        f"{bn_tr['pct_ge1_strict']:.0f}% of train carries a basic N; CYP2D6 "
        f"{d6['pct_ge1_strict']:.0f}% against {oth.pct_ge1_strict.min():.0f}-"
        f"{oth.pct_ge1_strict.max():.0f}% for the others",
     "file": "section4_basic_nitrogen_summary.csv"},
    {"section": "5 flexibility", "headline":
        f"{both_fl['pct_0-2 (rigid)'] + both_fl['pct_3-5']:.0f}% have <=5 rotatable bonds; "
        f"{int(both_fl['n_>10 (floppy)'])} of {len(AUD)} exceed 10; median "
        f"{both_fl['rotb_median']:.0f}",
     "file": "section5_flexibility_summary.csv"},
    {"section": "6 redundancy", "headline":
        f"0 conformer-dependent descriptors in any of the three feature sets; "
        f"{len(only_3d)} Mordred 3D descriptors available and none used",
     "file": "section6_feature_redundancy.csv"},
])
summary.to_csv(OUT / "audit_summary.csv", index=False)
print(summary.to_string(index=False))

V["meta"] = {
    "notebook": "41_dataset_audit_3d",
    "n_train": int(len(TR)), "n_blind": int(len(BL)),
    "rdkit": rdkit.__version__,
    "written_at_utc": datetime.now(timezone.utc).isoformat(),
    "nothing_computed": ["quantum chemistry", "conformer generation", "model training",
                         "submission"],
}
(OUT / "verdict.json").write_text(json.dumps(V, indent=2))
print(f"\n-> verdict.json ({len(V)} top-level keys)")

      section                                                                                                                                             headline                                file
       1 size                                                                                        median 24 HA, p99 34, max 65; 3 of 5655 compounds above 50 HA           section1_size_summary.csv
   2 elements                                                           9 elements, 0 compounds outside C/N/O/S/P/F/Cl/Br/I, 0 radicals, 21 non-neutral as written      section2_element_inventory.csv
     3 stereo 50% of train / 35% of blind have a centre, but only 18 of 750 blind compounds have every centre specified; canonicalisation preserves stereo exactly         section3_stereo_summary.csv
    4 basic N                                                                             19% of train carries a basic N; CYP2D6 37% against 13-14% for the others section4_basic_nitrogen_summary.csv
5 fle

### What this audit does not establish

Stated so nothing here is carried further than it goes.

- **It measures the compounds, not the usefulness of any descriptor.** Every number above is a
  property of the library. Whether a 3D or nitrogen-local block improves a model is a separate
  question, and this project's own record on that is poor: notebook `18` gave a model exactly the
  charge descriptor Q3's mechanism argues for and got a clean tie on all four isoforms, with the
  augmented arm's errors pool-typical rather than decorrelated.
- **The cost model in Section 1.3 is a scaling argument, not a timing.** No xtb or ORCA job was
  run, so the shares are relative, and they say the budget is not tail-dominated — not what it
  costs.
- **Conformer counts in Section 5 are combinatorial estimates**, `3^k` capped, not the output of a
  generator. A real ETKDG run with RMSD pruning will return fewer, and the ratio varies by compound.
- **Section 3's "no chirality angle" conclusion is about configuration being unstated, not about
  the compounds being achiral.** Half the training set has a centre. The data simply does not say
  which enantiomer was assayed, and that is a limit on what can be computed from it rather than a
  chemical fact about the molecules.
- **Q3's CYP2D6 enrichment is a property of which compounds were put on that assay.** It supports
  computing the block where the mechanism applies; it is not evidence the mechanism drives the
  measured potencies, and notebook `17`'s own effect sizes there were modest (partial Spearman
  ~0.177).

### Decisions taken without approval

Everything the brief did not pin down, with the reason and the alternative that was not taken.

In [32]:
record_decision(
    decision="Defined basic nitrogen as a specific SMARTS, validated on ten reference compounds, "
             "and carried a looser variant through every table as a sensitivity band",
    reason="The brief specifies basic nitrogen in prose -- 'aliphatic N, not amide, not aromatic, "
           "with a lone pair available' -- which does not determine a single pattern. The strict "
           "SMARTS is written out clause by clause and validated 10/10 before being applied to any "
           "real compound (four of the references are notebook 17's own CYP2D6 amines, reused "
           "verbatim). The genuinely contestable clause is whether aniline-type nitrogen counts, "
           "so the loose variant differs in exactly that clause and nothing else, and both are "
           "reported everywhere. Q3's CYP2D6 enrichment holds under both, which is the point of "
           "carrying them.",
    alternatives="Pick one SMARTS and report a single number (hides that the absolute level is "
                 "definition-dependent by about a factor of two); or use an external pKa tool "
                 "(outside the brief's pure-RDKit constraint, and notebook 17 already did it -- "
                 "its output is cross-checked against instead).",
    authority="the brief's prose definition, which needed an operational form")

record_decision(
    decision="Counted stereocentres with Chem.FindPotentialStereo rather than "
             "rdMolDescriptors.CalcNumAtomStereoCenters",
    reason="Part 0 established by direct test that in RDKit 2026.3.3 CalcNumAtomStereoCenters "
           "counts specified AND unspecified centres together, so it cannot answer the brief's "
           "assigned-versus-unassigned question -- which is the question Section 3 exists to "
           "settle. FindPotentialStereo reports the two separately. This also changed Section 6: "
           "ecfp4_narrow's stereocenter_count column is built from that same merged call, so it is "
           "a potential-centre count and is described as one.",
    alternatives="Use CalcNumAtomStereoCenters plus CalcNumUnspecifiedAtomStereoCenters and "
                 "subtract (gives the same answer, but silently depends on the semantic rather "
                 "than testing it).",
    authority="the brief's requirement to report assigned and unassigned counts separately")

record_decision(
    decision="Cross-checked the basic-nitrogen count against notebook 17's saved pH 7.4 "
             "protonation states, which the brief did not ask for",
    reason="A SMARTS count with no external reference is unfalsifiable. Notebook 17's "
           "charge_logp_features.csv is an independent instrument (Dimorphite-DL's pKa model) over "
           "the same 5,655 compounds and is read-only, so the check was free. It found a real "
           f"asymmetric disagreement -- {int(agree.loc[0, 'n_no_basic_N_but_cationic'])} compounds carry no basic "
           "nitrogen by this SMARTS yet protonate to +1 -- which is reported plainly and "
           "characterised structurally rather than averaged away, with the strict count framed "
           "as a lower bound.",
    alternatives=f"Report the SMARTS count alone (leaves a "
                 f"{100 - agree.loc[0, 'agreement_pct']:.0f}-point disagreement with this "
                 "project's own prior measurement undiscovered and uncited).",
    authority="none -- added because it strengthens a result the brief does rely on")

record_decision(
    decision="Reported rotatable bonds under RDKit's strict=True default as primary, with the "
             "non-strict count beside it",
    reason="The brief asks for a rotatable-bond distribution without naming a definition, and the "
           "two differ by about one bond per compound -- which enters a conformer budget as a "
           "multiplicative factor, not an additive one. Both are in the audit table and the "
           "summary so the budget can be re-derived under either.",
    alternatives="Report only the default (hides a real factor-level choice).",
    authority="the brief's conformer-protocol framing, which makes the choice consequential")

record_decision(
    decision="Added Section 1.3, a cost model, and Section 5's conformer-count table -- neither "
             "requested as such",
    reason="The brief asks what the size tail implies for a QM budget and says the conformer "
           "protocol cost is conformers x compounds. Answering either needs an explicit scaling "
           "assumption, so one is stated (heavy atoms to the power 2.5) and checked at 2.0, 3.0 "
           "and 4.0 so the conclusion does not rest on it. Both tables are counts under a stated "
           "assumption, not timings -- no job was run.",
    alternatives="Answer the budget question from percentiles alone (leaves 'does the tail "
                 "dominate' as an impression rather than a number).",
    authority="the brief's questions 2 and 5, which ask for budget implications")

record_decision(
    decision="Split the non-neutral compounds into permanent cations and carboxylates written "
             "deprotonated",
    reason="The brief asks how many compounds are not neutral as written. The raw count (21 "
           "training, 0 blind) mixes two different setup problems: a quaternary ammonium or "
           "cyanine has no neutral form and must be run as an ion, whereas a [O-] carboxylate is a "
           "drawing convention and the same compound could have been written neutral. Reporting "
           "one number would make the second look like the first.",
    alternatives="Report the bare count (accurate, and misleading about what it implies for a job "
                 "script).",
    authority="none -- the distinction is what makes the count actionable")

record_decision(
    decision="Named the eleven iodine-containing compounds although iodine is inside the brief's "
             "own allowed element set",
    reason="The brief asks for anything outside C/N/O/S/P/F/Cl/Br/I, and nothing is -- so that "
           "flag is empty. Iodine is nonetheless the heaviest element present and the one where an "
           "all-electron basis set stops being chooseable by default (a relativistic ECP is "
           "needed), which is the brief's own stated reason for caring about the element "
           "inventory. The compounds are named as context, explicitly not as a flag.",
    alternatives="Report the empty flag only (satisfies the letter of the brief and drops the one "
                 "element-driven setup consideration that actually applies here).",
    authority="none -- a deliberate addition in the spirit of the brief's stated reason")

print(f"{len(DECISIONS)} decisions recorded in decisions_taken.json\n")
for i, d in enumerate(DECISIONS, 1):
    print(f"{i}. {d['decision']}")
    print(f"   WHY: {d['reason']}")
    print(f"   ALTERNATIVES: {d['alternatives']}")
    print(f"   AUTHORITY: {d['authority']}\n")

7 decisions recorded in decisions_taken.json

1. Defined basic nitrogen as a specific SMARTS, validated on ten reference compounds, and carried a looser variant through every table as a sensitivity band
   WHY: The brief specifies basic nitrogen in prose -- 'aliphatic N, not amide, not aromatic, with a lone pair available' -- which does not determine a single pattern. The strict SMARTS is written out clause by clause and validated 10/10 before being applied to any real compound (four of the references are notebook 17's own CYP2D6 amines, reused verbatim). The genuinely contestable clause is whether aniline-type nitrogen counts, so the loose variant differs in exactly that clause and nothing else, and both are reported everywhere. Q3's CYP2D6 enrichment holds under both, which is the point of carrying them.
   ALTERNATIVES: Pick one SMARTS and report a single number (hides that the absolute level is definition-dependent by about a factor of two); or use an external pKa tool (outside the

### Scope check — fails loudly

Three tests, following notebooks `30`/`34`/`35`/`37`/`38`/`39`'s fixed version. **TEST 1** looks
only at *tracked* content changes under a protected path, so a pre-existing untracked directory
cannot raise a false positive (the bug notebook `29` shipped). **TEST 2** checks that nothing
inside a protected tree was written during this run, against the timestamp the setup cell recorded
at the top rather than one inferred from a file mtime at the bottom (the bug notebook `38`
shipped). **TEST 3** confirms the two notebooks carrying live submission code were never modified.
A fourth confirms no submission-shaped file was created anywhere, since this notebook has no
business producing one.

In [33]:
st_out = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True,
                        cwd=REPO_ROOT).stdout
print("git status --porcelain:")
print(st_out if st_out.strip() else "(clean)")

PROTECTED = [
    "data/", "src/",
    "notebooks/30_aid_full_retrain.ipynb", "outputs/30_aid_full_retrain/",
    "notebooks/32_deadzone_aid_retrain.ipynb", "outputs/32_deadzone_aid_retrain/",
    "notebooks/33_butina_cluster_split.ipynb", "outputs/33_butina_split/",
    "notebooks/34_butina_5x5_repeated_cv.ipynb", "outputs/34_butina_5x5/",
    "notebooks/35_butina_repeat3.ipynb", "outputs/35_butina_repeat3/",
    "notebooks/36_spread_corrected_nb32a.ipynb", "outputs/36_spread_corrected_nb32a/",
    "notebooks/37_butina_5x5_complete.ipynb", "outputs/37_butina_5x5_complete/",
    "notebooks/38_cyp3a4_placement_diagnostic.ipynb", "outputs/38_cyp3a4_placement_diagnostic/",
    "notebooks/39_ambiguity_and_shared_blend.ipynb", "outputs/39_ambiguity_and_shared_blend/",
    "notebooks/40_single_object_ensemble.ipynb", "outputs/40_single_object_ensemble/",
    "outputs/17_protonation_charge_check/", "outputs/23_compound_pool_overlap/",
    "outputs/mordred_pca/", "outputs/submissions/", "outputs/board_solved_calibration/",
    "scripts/",
]

tracked_violations = [l for l in st_out.splitlines()
                      if any(x in l[3:].strip() for x in PROTECTED) and l[:2].strip() != "??"]
written = []
for prot in PROTECTED:
    q = REPO_ROOT / prot
    if not q.exists():
        continue
    for fp in ([q] if q.is_file() else [x for x in q.rglob("*") if x.is_file()]):
        try:
            if fp.stat().st_mtime > RUN_STARTED_AT:
                written.append(str(fp.relative_to(REPO_ROOT)))
        except OSError:
            pass

print(f"\nTEST 1 -- tracked modifications under a protected path: {len(tracked_violations)}")
for t in tracked_violations:
    print(f"    VIOLATION: {t}")
print(f"TEST 2 -- files written inside a protected tree during this run: {len(written)}")
for t in written[:25]:
    print(f"    VIOLATION: {t}")

LIVE_SUBMISSION_NOTEBOOKS = ["notebooks/32_deadzone_aid_retrain.ipynb",
                             "notebooks/40_single_object_ensemble.ipynb"]
live_ok = True
for rel in LIVE_SUBMISSION_NOTEBOOKS:
    p = REPO_ROOT / rel
    ok = (not p.exists()) or p.stat().st_mtime <= RUN_STARTED_AT
    live_ok &= ok
    print(f"TEST 3 -- {rel} unmodified: {ok}")

made = sorted(p.relative_to(OUT).as_posix() for p in OUT.rglob("*") if p.is_file())
subm = [m for m in made if "submission" in m.lower()]
print(f"TEST 4 -- submission-shaped files created by this notebook: {len(subm)} {subm}")

if tracked_violations or written or not live_ok or subm:
    raise ValueError("scope violated -- investigate before trusting anything above")
print("\nconfirmed: no protected path modified, both live-submission notebooks untouched, "
      "no submission file created.")
print(f"\noutputs written by this notebook ({len(made)}):")
for m in made:
    print(f"  {m}")

git status --porcelain:
 M CLAUDE.md
 M docs/leaderboard_submissions.md
?? notebooks/35_butina_repeat3.ipynb
?? notebooks/36_spread_corrected_nb32a.ipynb
?? notebooks/37_butina_5x5_complete.ipynb
?? notebooks/38_cyp3a4_placement_diagnostic.ipynb
?? notebooks/39_ambiguity_and_shared_blend.ipynb
?? notebooks/40_single_object_ensemble.ipynb
?? notebooks/41_dataset_audit_3d.ipynb
?? outputs/35_butina_repeat3/
?? outputs/36_spread_corrected_nb32a/
?? outputs/37_butina_5x5_complete/
?? outputs/38_cyp3a4_placement_diagnostic/
?? outputs/39_ambiguity_and_shared_blend/
?? outputs/40_single_object_ensemble/
?? outputs/41_dataset_audit_3d/




TEST 1 -- tracked modifications under a protected path: 0
TEST 2 -- files written inside a protected tree during this run: 0
TEST 3 -- notebooks/32_deadzone_aid_retrain.ipynb unmodified: True
TEST 3 -- notebooks/40_single_object_ensemble.ipynb unmodified: True
TEST 4 -- submission-shaped files created by this notebook: 0 []

confirmed: no protected path modified, both live-submission notebooks untouched, no submission file created.

outputs written by this notebook (33):
  audit_summary.csv
  decisions_taken.json
  figures/basic_nitrogen_by_isoform.png
  figures/heavy_atom_distribution.png
  figures/rotatable_bond_distribution.png
  figures/stereochemistry_status.png
  part0_provenance.csv
  per_compound_audit.csv
  run_meta.json
  section1_cost_model.csv
  section1_isoform_tails.csv
  section1_large_compound_counts.csv
  section1_largest_compounds.csv
  section1_size_summary.csv
  section2_element_inventory.csv
  section2_formal_charge_summary.csv
  section2_iodine_compounds.csv
  se

In [34]:
print("=" * 84)
print("NOTEBOOK 41 COMPLETE -- audit only")
print("=" * 84)
print(f"""
  {len(AUD)} compounds audited ({len(TR)} train + {len(BL)} blind), 0 parse failures,
  0 warnings of any kind, 6 sections, 4 figures, {len(made)} output files.

  NOTHING quantum-mechanical computed. NO conformer generated. NO model trained. NO submission
  built, validated or sent. Pure RDKit and pandas throughout.

  THE FOUR ANSWERS, in one line each:
    chirality   SPLIT -- 3D shape is alive, chirality is not: only {int(bl_s['n_fully_specified'])} of {int(bl_s['n'])} blind
                compounds have every stereocentre specified.
    size tail   NOT dominant -- p99 {both_sz['p99']:.0f} HA, {int((AUD.heavy_atoms > 50).sum())} of {len(AUD)} compounds above 50 HA;
                a named handful of about ten compounds dominates, not an isoform.
    basic N     NO as a library-wide block ({bn_tr['pct_ge1_strict']:.0f}% coverage); YES for CYP2D6
                ({d6['pct_ge1_strict']:.0f}%, {d6['pct_ge1_strict'] / oth.pct_ge1_strict.mean():.1f}x the other three isoforms).
    redundancy  NO overlap -- 0 conformer-dependent descriptors in any existing feature set.

  No modelling decision is made or recommended here. The audit's purpose was to replace the
  descriptor plan's assumptions with measurements, and Q1 and Q2 both came back differently from
  what the plan assumed.
""")

NOTEBOOK 41 COMPLETE -- audit only

  5655 compounds audited (4905 train + 750 blind), 0 parse failures,
  0 warnings of any kind, 6 sections, 4 figures, 33 output files.

  NOTHING quantum-mechanical computed. NO conformer generated. NO model trained. NO submission
  built, validated or sent. Pure RDKit and pandas throughout.

  THE FOUR ANSWERS, in one line each:
    chirality   SPLIT -- 3D shape is alive, chirality is not: only 18 of 750 blind
                compounds have every stereocentre specified.
    size tail   NOT dominant -- p99 34 HA, 3 of 5655 compounds above 50 HA;
                a named handful of about ten compounds dominates, not an isoform.
    basic N     NO as a library-wide block (19% coverage); YES for CYP2D6
                (37%, 2.8x the other three isoforms).
    redundancy  NO overlap -- 0 conformer-dependent descriptors in any existing feature set.

  No modelling decision is made or recommended here. The audit's purpose was to replace the
  descriptor pla